# Does Script Representation Matter? Evidence from Three Cuneiform Languages
## Unified Three-Language Experiment Pipeline — EMNLP 2026
### Anonymous Authors

This notebook runs all preprocessing and experiments across **Elamite**, **Akkadian**, and **Sumerian**.  
Each experiment compares **Latin transliteration** vs **Unicode cuneiform** representations.

**Experiments:**
1. Embedding comparison (fastText, silhouette, morpheme coherence, graph analysis)
2. Word boundary inference (transitional probability)
3. POS classification (char n-gram LR, k-NN, BiLSTM) — two tagsets
4. Lemmatization (LSTM seq2seq)
5. LLM evaluation (Claude + GPT-4o, zero/few-shot) - just selection for elamite

## 0. Setup & Dependencies

In [1]:
!pip install -q gensim scikit-learn matplotlib seaborn pandas openpyxl regex torch umap-learn

import pandas as pd
import numpy as np
import regex as re
import json, os, time
from collections import Counter, defaultdict

from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (classification_report, f1_score,
                             silhouette_score, silhouette_samples)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = 'data/'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 98.4 MB/s eta 0:00:00
Mounted at /content/drive


## 1. POS Harmonization

Two tagsets evaluated in parallel:
- **Unified**: entity types (PN/DN/GN) kept as POS categories
- **Grammatical**: entities collapsed → PROPN, with separate NER tags

In [2]:

# ============================================================
# POS HARMONIZATION MAPS
# ============================================================

POS_UNIFIED_MAP = {
    'akk': {
        'N': 'NOUN', 'V': 'VERB', 'AJ': 'ADJ', 'AV': 'ADV',
        'PRP': 'ADP', 'DET': 'DET', 'CNJ': 'CONJ', 'MOD': 'MOD',
        'REL': 'REL', 'SBJ': 'SBJN', 'IP': 'INTJ',
        'PN': 'PN', 'DN': 'DN', 'GN': 'GN', 'CN': 'CN',
        'RN': 'RN', 'QN': 'QN', 'WN': 'WN', 'MN': 'MN',
        'AN': 'AN', 'FN': 'FN', 'TN': 'TN', 'LN': 'LN', 'ON': 'ON',
        'n': 'NUM', 'u': 'X', 'X': 'X',
    },
    'sux': {
        'N': 'NOUN', 'V/t': 'VERB', 'V/i': 'VERB', 'V': 'VERB',
        'AJ': 'ADJ', 'AV': 'ADV',
        'NU': 'NUM', 'IP': 'INTJ', 'QP': 'QP',
        'DN': 'DN', 'GN': 'GN', 'PN': 'PN', 'RN': 'RN',
        'SN': 'SN', 'TN': 'TN', 'WN': 'WN', 'MN': 'MN',
        'NA': 'X',
    },
    'elx': {
        'Noun': 'NOUN', 'Verb': 'VERB', 'ADJ': 'ADJ', 'other': 'OTHER',
        'PN': 'PN', 'PN-hyp': 'PN', 'GN': 'GN', 'DN': 'DN', 'Magic': 'OTHER',
    },
}

ENTITY_TAGS = {'PN', 'DN', 'GN', 'CN', 'RN', 'QN', 'WN', 'MN', 'AN', 'FN', 'TN', 'LN', 'ON', 'SN', 'QP'}

def map_pos_unified(pos_raw, lang):
    return POS_UNIFIED_MAP.get(lang, {}).get(pos_raw, 'X')

def map_pos_grammatical(pos_unified):
    return 'PROPN' if pos_unified in ENTITY_TAGS else pos_unified

def map_ner_tag(pos_unified):
    return pos_unified if pos_unified in ENTITY_TAGS else 'O'

print("POS harmonization maps loaded.")
print(f"  Akkadian: {len(POS_UNIFIED_MAP['akk'])} mappings")
print(f"  Sumerian: {len(POS_UNIFIED_MAP['sux'])} mappings")
print(f"  Elamite:  {len(POS_UNIFIED_MAP['elx'])} mappings")

POS harmonization maps loaded.
  Akkadian: 27 mappings
  Sumerian: 18 mappings
  Elamite:  9 mappings


## 2. Sign Lists & Unicode Conversion

In [3]:
# ============================================================
# LOAD & MERGE SIGN LISTS
# ============================================================
sign_list = pd.read_json(
    'https://raw.githubusercontent.com/situx/Nuolenna/master/sign_list.json', orient='index'
)
sign_list.columns = ['unicode']
sign_list['sign'] = sign_list.index.tolist()
sign_list = sign_list[['sign', 'unicode']].reset_index(drop=True)

akkademia = pd.read_csv(
    'https://raw.githubusercontent.com/gaigutherz/Akkademia/master/cuneiform_to_unicode_fixed.csv'
)
merged = pd.merge(sign_list, akkademia, on=['sign', 'unicode'], how='outer')
sign_dict = dict(zip(merged['sign'].astype(str), merged['unicode'].astype(str)))

# Load manual corrections if available
UNMATCHED_PATH_1 = BASE_PATH + 'unmatchednew_AAedit - unmatchednew.csv'
UNMATCHED_PATH_2 = BASE_PATH + 'unmatchednew - solonew.csv'
try:
    unmatched = pd.read_csv(UNMATCHED_PATH_1)[['unmatched_sign','use']].dropna()
    unmatched2 = pd.read_csv(UNMATCHED_PATH_2)[['value', 'SIGN']].dropna()
    manual_dict = dict(zip(unmatched['unmatched_sign'], unmatched['use']))
    manual_dict2 = dict(zip(unmatched2['value'].str.strip("[]' "), unmatched2['SIGN']))
    sign_dict.update(manual_dict)
    sign_dict.update(manual_dict2)
    print(f"Manual corrections loaded: {len(manual_dict)} + {len(manual_dict2)}")
except FileNotFoundError:
    print("Manual correction files not found — using base sign lists only.")

print(f"Total sign mappings: {len(sign_dict)}")

Manual corrections loaded: 114 + 22
Total sign mappings: 18271


In [4]:
# ============================================================
# UNICODE CONVERSION FUNCTIONS
# ============================================================

def normalize_transliteration(text, lang='akk'):
    if pd.isna(text) or text == '':
        return ''
    s = str(text)
    s = re.sub(r'\{[^}]*\}', '', s)          # Remove determinatives
    s = s.replace('-', ' ').replace('.', ' ')  # Sign separators → space
    for ch in ['[', ']', '#', '!', '?', '*', '(', ')']:
        s = s.replace(ch, '')
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def transliteration_to_unicode(text, sign_dict, lang='akk'):
    normalized = normalize_transliteration(text, lang)
    if not normalized:
        return '', []
    tokens = normalized.split()
    out, unmatched = [], []
    for tok in tokens:
        if tok in sign_dict:
            out.append(sign_dict[tok])
        elif tok.lower() in sign_dict:
            out.append(sign_dict[tok.lower()])
        elif tok.upper() in sign_dict:
            out.append(sign_dict[tok.upper()])
        else:
            out.append(tok)
            if re.search(r'\p{Latin}', tok):
                unmatched.append(tok)
    return ' '.join(out), unmatched


def convert_column_to_unicode(forms, sign_dict, lang='akk'):
    results = forms.apply(lambda x: transliteration_to_unicode(x, sign_dict, lang))
    unicode_col = results.apply(lambda x: x[0])
    unmatched_col = results.apply(lambda x: x[1])
    n_clean = (unmatched_col.apply(len) == 0).sum()
    rate = n_clean / len(forms) if len(forms) > 0 else 0
    return unicode_col, rate

print("Unicode conversion functions defined.")

Unicode conversion functions defined.


## 3. Load All Three Languages

In [5]:
# ============================================================
# LANGUAGE-SPECIFIC LOADERS
# ============================================================

def load_akkadian(csv_path):
    df = pd.read_csv(csv_path, low_memory=False)
    df = df[df['pos'].notna() & (df['pos'] != 'u')].copy()
    df = df[df['form'].notna() & (df['form'] != 'x') & (df['form'] != 'X')].copy()
    out = pd.DataFrame({
        'token_id': range(len(df)),
        'text_id': df['id_text'].values,
        'form_latin': df['form'].values,
        'pos_raw': df['pos'].values,
        'lemma': df['cf'].values,
        'gloss': df['gw'].values,
        'language': 'akk',
    })
    out['pos_unified'] = out['pos_raw'].apply(lambda x: map_pos_unified(x, 'akk'))
    out['pos_grammatical'] = out['pos_unified'].apply(map_pos_grammatical)
    out['ner_tag'] = out['pos_unified'].apply(map_ner_tag)
    return out


def load_sumerian(csv_path):
    df = pd.read_csv(csv_path)
    df = df[df['pos'].notna() & (df['pos'] != '') & (df['pos'].astype(str) != 'nan')].copy()
    df = df[df['form'].notna()].copy()
    out = pd.DataFrame({
        'token_id': range(len(df)),
        'text_id': df['id_text'].values,
        'form_latin': df['form'].values,
        'pos_raw': df['pos'].values,
        'lemma': df['cf'].values,
        'gloss': df['gw'].values,
        'language': 'sux',
    })
    out['pos_unified'] = out['pos_raw'].apply(lambda x: map_pos_unified(x, 'sux'))
    out['pos_grammatical'] = out['pos_unified'].apply(map_pos_grammatical)
    out['ner_tag'] = out['pos_unified'].apply(map_ner_tag)
    return out


def add_unicode_representations(df, sign_dict):
    df['form_unicode'], rate = convert_column_to_unicode(
        df['form_latin'], sign_dict, df['language'].iloc[0]
    )
    df['form_unicode_nospace'] = df['form_unicode'].str.replace(' ', '')
    df['unicode_clean'] = ~df['form_unicode'].apply(
        lambda x: bool(re.search(r'\p{Latin}', str(x))) if pd.notna(x) else True
    )
    print(f"  Unicode conversion rate: {rate:.1%} ({df['unicode_clean'].sum()}/{len(df)})")
    return df


def dataset_summary(df, name=''):
    print(f"\n{'='*60}")
    print(f"Dataset: {name}")
    print(f"{'='*60}")
    print(f"Tokens:     {len(df):,}")
    print(f"Texts:      {df['text_id'].nunique():,}")
    print(f"Vocab:      {df['form_latin'].nunique():,}")
    print(f"Lemmas:     {df['lemma'].nunique():,}")
    print(f"\nUnified POS distribution:")
    for pos, cnt in df['pos_unified'].value_counts().head(12).items():
        print(f"  {pos:8s}: {cnt:7,} ({cnt/len(df)*100:5.1f}%)")
    print(f"\nGrammatical POS distribution:")
    for pos, cnt in df['pos_grammatical'].value_counts().items():
        print(f"  {pos:8s}: {cnt:7,} ({cnt/len(df)*100:5.1f}%)")
    ner_counts = df[df['ner_tag'] != 'O']['ner_tag'].value_counts()
    print(f"\nEntities: {(df['ner_tag'] != 'O').sum():,} ({(df['ner_tag'] != 'O').mean()*100:.1f}%)")
    for tag, cnt in ner_counts.head(8).items():
        print(f"  {tag:6s}: {cnt:7,}")

print("Loaders defined.")

Loaders defined.


In [6]:
# ============================================================
# LOAD DATA
# ============================================================
AKK_PATH = BASE_PATH + 'alltexts_AKK.csv'
SUX_PATH = BASE_PATH + 'alltexts_SUX.csv'

print("Loading Akkadian...")
akk = load_akkadian(AKK_PATH)
dataset_summary(akk, 'Akkadian')

print("\nLoading Sumerian...")
sux = load_sumerian(SUX_PATH)
dataset_summary(sux, 'Sumerian')

print("\nConverting Akkadian to Unicode...")
akk = add_unicode_representations(akk, sign_dict)

print("\nConverting Sumerian to Unicode...")
sux = add_unicode_representations(sux, sign_dict)

datasets = {'akk': akk, 'sux': sux}

Loading Akkadian...

Dataset: Akkadian
Tokens:     1,255,669
Texts:      13,720
Vocab:      123,001
Lemmas:     28,953

Unified POS distribution:
  NOUN    : 495,937 ( 39.5%)
  VERB    : 132,956 ( 10.6%)
  ADP     : 121,008 (  9.6%)
  NUM     : 104,830 (  8.3%)
  PN      :  69,816 (  5.6%)
  X       :  60,665 (  4.8%)
  ADJ     :  55,973 (  4.5%)
  DET     :  40,775 (  3.2%)
  DN      :  25,547 (  2.0%)
  CONJ    :  21,920 (  1.7%)
  CN      :  21,118 (  1.7%)
  MOD     :  20,493 (  1.6%)

Grammatical POS distribution:
  NOUN    : 495,937 ( 39.5%)
  PROPN   : 158,407 ( 12.6%)
  VERB    : 132,956 ( 10.6%)
  ADP     : 121,008 (  9.6%)
  NUM     : 104,830 (  8.3%)
  X       :  60,665 (  4.8%)
  ADJ     :  55,973 (  4.5%)
  DET     :  40,775 (  3.2%)
  CONJ    :  21,920 (  1.7%)
  MOD     :  20,493 (  1.6%)
  REL     :  17,780 (  1.4%)
  INTJ    :  12,497 (  1.0%)
  ADV     :  11,626 (  0.9%)
  SBJN    :     802 (  0.1%)

Entities: 158,407 (12.6%)
  PN    :  69,816
  DN    :  25,547
  CN  

## 4. Build Document Corpora

In [7]:
# ============================================================
# BUILD PER-LANGUAGE DOCUMENT DICTS
# ============================================================
doc_corpora = {}
for lang, df in datasets.items():
    docs_latin = {tid: ' '.join(g['form_latin'].dropna().astype(str))
                  for tid, g in df.groupby('text_id')}
    docs_unicode = {tid: ' '.join(g['form_unicode'].dropna().astype(str))
                    for tid, g in df.groupby('text_id')}
    doc_corpora[lang] = {'latin': docs_latin, 'unicode': docs_unicode}
    print(f"{lang}: {len(docs_latin)} documents")

akk: 13720 documents
sux: 394 documents


In [8]:
# ── Load Elamite (original pipeline — exact match) ──
DICT_PATH = BASE_PATH + 'Elamite_Lemma-base-draft.xlsx'
NASU_PATH = BASE_PATH + 'UnTN-Nasu texts Word-level.csv'

# 1. Load dictionary from Excel tabs
all_sheets = pd.read_excel(DICT_PATH, sheet_name=None)
target_tabs = ['ADJ', 'Noun', 'Verb', 'other', 'PN', 'PN-hyp', 'GN', 'DN', 'Magic']
columns_to_keep = [
    'transliteration', 'sorting', 'period', 'base',
    'logogram', 'morpheme_1', 'morpheme_2', 'morpheme_3',
    'sense_hk', 'certainty-weight_hk', 'sense_hk_qid',
    'certainty-weight_MEGA', 'sense_MEGA_qid', 'POS', 'number', 'person'
]
combined_list = []
for name in target_tabs:
    if name in all_sheets:
        df = all_sheets[name]
        df['category'] = name
        existing_cols = [c for c in columns_to_keep if c in df.columns]
        combined_list.append(df[existing_cols + ['category']])
final_dictionary = pd.concat(combined_list, ignore_index=True)
print(f"Dictionary: {len(final_dictionary)} entries from {len(combined_list)} tabs")
print(f"POS: {final_dictionary['category'].value_counts().to_dict()}")

# 2. First-pass preprocessing
to_unicode = final_dictionary.copy()
to_unicode['transliteration original'] = to_unicode['transliteration']
to_unicode['transliteration'] = to_unicode['transliteration'].astype(str).fillna('')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('-', ' ')
for ch in ['_', '[', ']', '*', '!', '?', '/', ',', ':', ';', '^', '`']:
    to_unicode['transliteration'] = to_unicode['transliteration'].str.replace(ch, '', regex=False)
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('.', ' ', regex=False)
to_unicode['transliteration'] = to_unicode['transliteration'].str.lower()
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace(r'~.*?……', '……', regex=True)
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('X', '', regex=False)
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('……', ' ', regex=False)

def replace_with_unicode(text):
    return ' '.join([sign_dict.get(s, s) for s in text.split()])

to_unicode['unicode'] = to_unicode['transliteration'].apply(replace_with_unicode)
to_unicode['roman'] = to_unicode['unicode'].apply(
    lambda x: any(bool(re.search(r'\p{Latin}', w)) for w in x.split()))
print(f"After first pass: {(~to_unicode['roman']).sum()}/{len(to_unicode)} clean "
      f"({(~to_unicode['roman']).sum()/len(to_unicode)*100:.1f}%)")

# 3. Second-pass conversion — build v2 dict (matches original exactly)
unicode_dict_v2 = dict(zip(merged['sign'].astype(str), merged['unicode'].astype(str)))
unicode_dict_v2.update(manual_dict)
unicode_dict_v2.update(manual_dict2)

def normalize_for_unmatched_pass(s):
    if pd.isna(s): return ""
    s = str(s)
    s = re.sub(r"\(\s*md\s*\)", " m d ", s)
    s = re.sub(r"[.,:;!?()\[\]{}<>\\\\\"\"\"''/\\|*^`~]", " ", s)
    s = re.sub(r"[-\u2013\u2014]", " ", s)
    s = re.sub(r"([A-Za-z])(\d)([A-Za-z])", r"\1\2 \3", s)
    s = re.sub(r"(\d)([A-Za-z])", r"\1 \2", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def final_unmatched_conversion_pass(unicode_text):
    txt = normalize_for_unmatched_pass(unicode_text)
    tokens = txt.split()
    out, still_unmatched = [], []
    for tok in tokens:
        if tok == "md" and "m" in unicode_dict_v2 and "d" in unicode_dict_v2 and "md" not in unicode_dict_v2:
            out.append(unicode_dict_v2["m"]); out.append(unicode_dict_v2["d"]); continue
        if tok in unicode_dict_v2:
            out.append(unicode_dict_v2[tok])
        else:
            out.append(tok)
            if re.search(r'\p{Latin}', tok): still_unmatched.append(tok)
    return " ".join(out), still_unmatched

tmp = to_unicode['unicode'].apply(final_unmatched_conversion_pass)
to_unicode['transliteration'] = to_unicode['transliteration original'].apply(normalize_for_unmatched_pass)
to_unicode['roman'] = tmp.apply(lambda x: len(x[1]) > 0)
to_unicode['unicode'] = tmp.apply(lambda x: x[0])

dict_clean = to_unicode[to_unicode['roman'] == False].copy()
print(f"Clean entries: {len(dict_clean)}/{len(to_unicode)} ({len(dict_clean)/len(to_unicode)*100:.1f}%)")

# 4. Three representations
dict_clean['repr_latin'] = dict_clean['transliteration']
dict_clean['repr_unicode'] = dict_clean['unicode']
dict_clean['repr_unicode_nospace'] = dict_clean['unicode'].str.replace(' ', '')

print('=== Samples ===')
for _, r in dict_clean[['transliteration original','repr_latin','repr_unicode','category']].head(5).iterrows():
    print(f"  {r['transliteration original']:30s} → Latin: {r['repr_latin']:25s} Unicode: {r['repr_unicode']:30s} [{r['category']}]")

# 5. Convert to unified schema
elx = pd.DataFrame({
    'token_id': range(len(dict_clean)),
    'text_id': 'elx_dict',
    'form_latin': dict_clean['transliteration'].values,
    'form_unicode': dict_clean['unicode'].values,
    'form_unicode_nospace': dict_clean['unicode'].str.replace(' ', '').values,
    'pos_raw': dict_clean['category'].values,
    'lemma': dict_clean['base'].values,
    'gloss': dict_clean['sense_hk'].values,
    'language': 'elx',
    'unicode_clean': True,
})
# Preserve morpheme_1 for coherence calculation
if 'morpheme_1' in dict_clean.columns:
    elx['morpheme_1'] = dict_clean['morpheme_1'].values

elx['pos_unified'] = elx['pos_raw'].apply(lambda x: map_pos_unified(x, 'elx'))
elx['pos_grammatical'] = elx['pos_unified'].apply(map_pos_grammatical)
elx['ner_tag'] = elx['pos_unified'].apply(map_ner_tag)
datasets['elx'] = elx
dataset_summary(elx, 'Elamite (Dictionary)')

# 6. Load Nasu corpus for documents (original conversion: replace_with_unicode → final_unmatched_conversion_pass)
nasu_df = pd.read_csv(NASU_PATH)
nasu_df['translit_clean'] = nasu_df['transliteration'].apply(normalize_for_unmatched_pass)

def convert_nasu_to_unicode(text):
    if pd.isna(text) or text == '': return ''
    return final_unmatched_conversion_pass(replace_with_unicode(text))[0]

nasu_df['unicode'] = nasu_df['translit_clean'].apply(convert_nasu_to_unicode)

doc_corpora['elx'] = {
    'latin': {tid: ' '.join(g['translit_clean'].dropna().astype(str))
              for tid, g in nasu_df.groupby('id_text')},
    'unicode': {tid: ' '.join(g['unicode'].dropna().astype(str))
                for tid, g in nasu_df.groupby('id_text')},
}
print(f"\nNasu corpus: {len(nasu_df)} tokens, {nasu_df['id_text'].nunique()} documents")


Dictionary: 15601 entries from 8 tabs
POS: {'PN': 6060, 'Noun': 4306, 'GN': 1756, 'Verb': 1711, 'other': 787, 'ADJ': 614, 'DN': 337, 'PN-hyp': 30}
After first pass: 13302/15601 clean (85.3%)
Clean entries: 13383/15601 (85.8%)
=== Samples ===
  a-a                            → Latin: a a                       Unicode: 𒀀 𒀀                            [ADJ]
  h.hi-bat-tin-na                → Latin: h hi bat tin na           Unicode: 𒀸 𒄭 𒁁 𒁷 𒈾                      [ADJ]
  v.ha-ak-qa-man-nu-iš-ši-ya     → Latin: v ha ak qa man nu iš ši ya Unicode: 𒁹 𒄩 𒀝 𒋡 𒌋𒌋 𒉡 𒅖 𒅆 𒉿             [ADJ]
  am-mín-nu                      → Latin: am mín nu                 Unicode: 𒄠 𒊩 𒉡                          [ADJ]
  te-man?-na?-na                 → Latin: te man na na              Unicode: 𒋼 𒌋𒌋 𒈾 𒈾                       [ADJ]

Dataset: Elamite (Dictionary)
Tokens:     13,383
Texts:      1
Vocab:      13,189
Lemmas:     748

Unified POS distribution:
  PN      :   5,012 ( 37.5%)
  NOUN    :   3,852 ( 28.8%)
  V

## Rare Token Analysis

In [9]:
# ============================================================
# RARE/FREQUENT TOKEN ANALYSIS
# ============================================================
# Does Unicode help more on rare tokens?
# Split POS classification results by token frequency bucket.
# Prerequisites: cells 1-13 (data loading), MIN_CLASS_COUNT=20
# ============================================================

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from collections import Counter
import numpy as np

SEED = 42
MIN_CLASS_COUNT = 20

for lang, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  RARE/FREQUENT ANALYSIS — {lang.upper()}")
    print(f"{'='*60}")

    # Count token frequencies
    freq = Counter(df['form_latin'].astype(str).values)

    # Use unified POS
    counts = df['pos_unified'].value_counts()
    valid = counts[counts >= MIN_CLASS_COUNT].index.tolist()
    task_df = df[df['pos_unified'].isin(valid)].copy()

    if len(task_df) > 10000:
        task_df = task_df.sample(10000, random_state=SEED)

    texts_l = task_df['form_latin'].astype(str).values
    texts_u = task_df['form_unicode'].astype(str).values
    labels = task_df['pos_unified'].values

    # Assign frequency bucket to each token
    token_freqs = np.array([freq[t] for t in texts_l])
    q25 = np.percentile(token_freqs, 25)
    q75 = np.percentile(token_freqs, 75)

    buckets = np.where(token_freqs <= q25, 'rare',
              np.where(token_freqs >= q75, 'frequent', 'medium'))

    print(f"\n  Frequency thresholds: rare ≤ {q25:.0f}, frequent ≥ {q75:.0f}")
    print(f"  Bucket sizes: rare={np.sum(buckets=='rare')}, "
          f"medium={np.sum(buckets=='medium')}, "
          f"frequent={np.sum(buckets=='frequent')}")

    # Train models and get per-token predictions
    vec_l = CountVectorizer(analyzer='char', ngram_range=(1, 4))
    vec_u = CountVectorizer(analyzer='char', ngram_range=(1, 4))
    X_l = vec_l.fit_transform(texts_l)
    X_u = vec_u.fit_transform(texts_u)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    preds_l = np.empty(len(labels), dtype=labels.dtype)
    preds_u = np.empty(len(labels), dtype=labels.dtype)

    for train_idx, test_idx in cv.split(X_l, labels):
        clf_l = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
        clf_u = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
        clf_l.fit(X_l[train_idx], labels[train_idx])
        clf_u.fit(X_u[train_idx], labels[train_idx])
        preds_l[test_idx] = clf_l.predict(X_l[test_idx])
        preds_u[test_idx] = clf_u.predict(X_u[test_idx])

    # Report F1 by bucket
    print(f"\n  {'Bucket':<12s} {'N':>6s} {'Latin F1':>10s} {'Unicode F1':>12s} {'Δ':>8s} {'Winner':>10s}")
    print(f"  {'-'*12} {'-'*6} {'-'*10} {'-'*12} {'-'*8} {'-'*10}")

    for bucket in ['rare', 'medium', 'frequent']:
        mask = buckets == bucket
        if mask.sum() < 20:
            continue

        f1_l = f1_score(labels[mask], preds_l[mask], average='macro', zero_division=0)
        f1_u = f1_score(labels[mask], preds_u[mask], average='macro', zero_division=0)
        delta = f1_u - f1_l
        winner = 'Unicode' if delta > 0.005 else ('Latin' if delta < -0.005 else 'tie')

        print(f"  {bucket:<12s} {mask.sum():>6d} {f1_l:>10.4f} {f1_u:>12.4f} {delta:>+8.4f} {winner:>10s}")

    # Also report by sign count for Unicode
    sign_counts = np.array([len(str(t).split()) for t in texts_u])

    print(f"\n  By sign count:")
    print(f"  {'Signs':<12s} {'N':>6s} {'Latin F1':>10s} {'Unicode F1':>12s} {'Δ':>8s} {'Winner':>10s}")
    print(f"  {'-'*12} {'-'*6} {'-'*10} {'-'*12} {'-'*8} {'-'*10}")

    for lo, hi, label in [(1, 1, '1 sign'), (2, 2, '2 signs'), (3, 3, '3 signs'),
                           (4, 5, '4-5 signs'), (6, 100, '6+ signs')]:
        mask = (sign_counts >= lo) & (sign_counts <= hi)
        if mask.sum() < 20:
            continue

        f1_l = f1_score(labels[mask], preds_l[mask], average='macro', zero_division=0)
        f1_u = f1_score(labels[mask], preds_u[mask], average='macro', zero_division=0)
        delta = f1_u - f1_l
        winner = 'Unicode' if delta > 0.005 else ('Latin' if delta < -0.005 else 'tie')

        print(f"  {label:<12s} {mask.sum():>6d} {f1_l:>10.4f} {f1_u:>12.4f} {delta:>+8.4f} {winner:>10s}")

    # Complementary analysis by bucket
    latin_correct = preds_l == labels
    unicode_correct = preds_u == labels
    latin_only = latin_correct & ~unicode_correct
    unicode_only = ~latin_correct & unicode_correct

    print(f"\n  Complementary tokens by frequency:")
    print(f"  {'Bucket':<12s} {'N':>6s} {'L-only':>8s} {'U-only':>8s} {'Compl%':>8s}")
    print(f"  {'-'*12} {'-'*6} {'-'*8} {'-'*8} {'-'*8}")

    for bucket in ['rare', 'medium', 'frequent']:
        mask = buckets == bucket
        if mask.sum() < 20:
            continue
        l_only = latin_only[mask].sum()
        u_only = unicode_only[mask].sum()
        compl_pct = (l_only + u_only) / mask.sum() * 100

        print(f"  {bucket:<12s} {mask.sum():>6d} {l_only:>8d} {u_only:>8d} {compl_pct:>7.1f}%")


  RARE/FREQUENT ANALYSIS — AKK

  Frequency thresholds: rare ≤ 26, frequent ≥ 6729
  Bucket sizes: rare=2507, medium=4938, frequent=2555

  Bucket            N   Latin F1   Unicode F1        Δ     Winner
  ------------ ------ ---------- ------------ -------- ----------
  rare           2507     0.4053       0.2583  -0.1471      Latin
  medium         4938     0.6392       0.5884  -0.0508      Latin
  frequent       2555     0.3353       0.3321  -0.0033        tie

  By sign count:
  Signs             N   Latin F1   Unicode F1        Δ     Winner
  ------------ ------ ---------- ------------ -------- ----------
  1 sign         4303     0.5640       0.4823  -0.0817      Latin
  2 signs        2426     0.6418       0.5527  -0.0891      Latin
  3 signs        1724     0.5014       0.4288  -0.0725      Latin
  4-5 signs      1081     0.4098       0.3331  -0.0766      Latin
  6+ signs         91     0.4060       0.3977  -0.0084      Latin

  Complementary tokens by frequency:
  Bucket     

## Cross Language Transfer

In [10]:
# ============================================================
# HITTITE CROSS-LANGUAGE TRANSFER
# ============================================================
# Paste this cell into ThreeLanguagePipeline_v2 AFTER cells 1-17
# (i.e., after data loading & sign list setup).
# Requires: sign_dict, doc_corpora (with akk/sux/elx) already loaded.
# ============================================================

import unicodedata
import random
from collections import Counter

# ── 1. Hittite-specific preprocessing & Unicode conversion ──

HITT_PATH = BASE_PATH + '7000_hitt_txts_wGloss.csv'

def normalize_for_lookup_hitt(tok):
    """Normalize Hittite characters for sign list lookup."""
    t = tok
    t = t.replace('ḫ', 'h').replace('Ḫ', 'H')
    t = ''.join(c for c in unicodedata.normalize('NFD', t)
                if unicodedata.category(c) != 'Mn')
    t = t.replace('~', '').replace('˽', '')
    t = t.replace('⸢', '').replace('⸣', '')
    return t

def lookup_sign_hitt(tok):
    """Try multiple normalization strategies."""
    for t in [tok, tok.lower(), tok.upper()]:
        if t in sign_dict: return sign_dict[t]
    n = normalize_for_lookup_hitt(tok)
    for t in [n, n.lower(), n.upper()]:
        if t in sign_dict: return sign_dict[t]
    stripped = re.sub(r'[₀₁₂₃₄₅₆₇₈₉]+$', '', n)
    for t in [stripped, stripped.lower(), stripped.upper()]:
        if t in sign_dict: return sign_dict[t]
    stripped2 = re.sub(r'\d+$', '', n)
    for t in [stripped2, stripped2.lower(), stripped2.upper()]:
        if t in sign_dict: return sign_dict[t]
    return None

def hittite_translit_to_signs(translit):
    """Convert Hittite transliteration to list of Unicode signs.

    Preprocessing per Zenodo documentation:
    1. Replace { } with whitespace
    2. Remove [ ] and ⸢ ⸣ (no whitespace)
    3. Convert to Unicode signs
    """
    t = str(translit)
    t = t.replace('{', ' ').replace('}', ' ')
    t = t.replace('[', '').replace(']', '')
    t = t.replace('⸢', '').replace('⸣', '')
    t = t.replace('-', ' ').replace('.', ' ')
    for ch in ['#', '!', '?', '*', '(', ')', '°', '½']:
        t = t.replace(ch, '')
    t = re.sub(r'\s+', ' ', t).strip()

    signs = []
    for tok in t.split():
        tok = tok.strip()
        if not tok: continue
        uni = lookup_sign_hitt(tok)
        if uni and str(uni) != 'nan':
            signs.append(uni)
    return signs

# ── 2. Load Hittite data ──

print("Loading Hittite data...")
hitt_raw = pd.read_csv(HITT_PATH)
hitt_raw = hitt_raw[hitt_raw['translit'].notna() & (hitt_raw['translit'] != '…')].copy()
print(f"  Raw tokens: {len(hitt_raw):,}")
print(f"  Texts: {hitt_raw['txtid'].nunique():,}")

# Convert to Unicode and build segmented documents
hitt_docs_segmented = {}
total_words = 0
total_converted = 0

for txtid, group in hitt_raw.groupby('txtid'):
    words = []
    for _, row in group.iterrows():
        signs = hittite_translit_to_signs(row['translit'])
        if signs:
            words.append(signs)
            total_converted += 1
        total_words += 1
    if len(words) >= 2:
        hitt_docs_segmented[txtid] = words

print(f"  Segmented documents: {len(hitt_docs_segmented):,}")
print(f"  Words converted: {total_converted:,}/{total_words:,} ({total_converted/total_words*100:.1f}%)")
total_signs = sum(sum(len(w) for w in d) for d in hitt_docs_segmented.values())
unique_signs = set(s for d in hitt_docs_segmented.values() for w in d for s in w)
print(f"  Total signs: {total_signs:,}")
print(f"  Unique Unicode signs: {len(unique_signs)}")

# Also build Unicode doc strings for doc_corpora compatibility
hitt_docs_unicode = {}
for txtid, group in hitt_raw.groupby('txtid'):
    unicode_words = []
    for _, row in group.iterrows():
        signs = hittite_translit_to_signs(row['translit'])
        if signs:
            unicode_words.append(' '.join(signs))
    if unicode_words:
        hitt_docs_unicode[txtid] = ' '.join(unicode_words)

# Build Latin doc strings
hitt_docs_latin = {}
for txtid, group in hitt_raw.groupby('txtid'):
    hitt_docs_latin[txtid] = ' '.join(group['translit'].dropna().astype(str))

doc_corpora['hit'] = {'latin': hitt_docs_latin, 'unicode': hitt_docs_unicode}
print(f"\n  Added 'hit' to doc_corpora: {len(hitt_docs_latin)} docs")

# ── 3. Build segmented docs for AKK/SUX/ELX ──
# (same format as Hittite: dict of {doc_id: [[sign, sign], [sign, sign, sign], ...]})

def unicode_docs_to_segmented(unicode_doc_dict):
    """Convert space-separated Unicode doc strings to segmented format."""
    segmented = {}
    for tid, doc in unicode_doc_dict.items():
        if not doc or not doc.strip():
            continue
        words = doc.split()
        word_signs = []
        for word in words:
            signs = [ch for ch in word if ch.strip()]
            if signs:
                word_signs.append(signs)
        if len(word_signs) >= 2:
            segmented[tid] = word_signs
    return segmented

all_segmented = {}
for lang in ['akk', 'sux', 'elx']:
    if lang in doc_corpora:
        all_segmented[lang] = unicode_docs_to_segmented(doc_corpora[lang]['unicode'])
        print(f"  {lang.upper()}: {len(all_segmented[lang])} segmented docs")

all_segmented['hit'] = hitt_docs_segmented
print(f"  HIT: {len(all_segmented['hit'])} segmented docs")

# ── 4. TP functions ──

def compute_tp(doc_ids, docs):
    bi, uni = Counter(), Counter()
    for did in doc_ids:
        s = [sign for w in docs[did] for sign in w]
        for i in range(len(s)):
            uni[s[i]] += 1
            if i < len(s) - 1:
                bi[(s[i], s[i+1])] += 1
    return {k: c / uni[k[0]] for k, c in bi.items()}

def evaluate_tp(doc_ids, docs, tp, theta):
    tc = fc = fnc = 0
    for did in doc_ids:
        words = docs[did]
        s = [sign for w in words for sign in w]
        gold = set(); pos = 0
        for w in words:
            pos += len(w)
            gold.add(pos)
        gold.discard(pos)
        pred = {i+1 for i in range(len(s)-1)
                if tp.get((s[i], s[i+1]), 0) < theta}
        tc += len(pred & gold)
        fc += len(pred - gold)
        fnc += len(gold - pred)
    p = tc / (tc + fc) if (tc + fc) else 0
    r = tc / (tc + fnc) if (tc + fnc) else 0
    f1 = 2 * p * r / (p + r) if (p + r) else 0
    return f1, p, r

def find_best_threshold(doc_ids, docs, tp):
    best_f1, best_theta = 0, 0.5
    for theta in np.arange(0.05, 0.99, 0.05):
        f1, _, _ = evaluate_tp(doc_ids, docs, tp, theta)
        if f1 > best_f1:
            best_f1, best_theta = f1, theta
    for theta in np.arange(max(0.01, best_theta - 0.1),
                            min(0.99, best_theta + 0.1), 0.01):
        f1, _, _ = evaluate_tp(doc_ids, docs, tp, theta)
        if f1 > best_f1:
            best_f1, best_theta = f1, theta
    f1, p, r = evaluate_tp(doc_ids, docs, tp, best_theta)
    return {'f1': f1, 'precision': p, 'recall': r, 'threshold': best_theta}

# ── 5. Full 4×4 cross-language transfer ──

lang_names = {'akk': 'Akkadian', 'sux': 'Sumerian', 'elx': 'Elamite', 'hit': 'Hittite'}
languages = [l for l in ['akk', 'sux', 'elx', 'hit'] if l in all_segmented]

print(f"\n{'='*70}")
print(f"  CROSS-LANGUAGE WORD BOUNDARY TRANSFER (4 languages)")
print(f"{'='*70}")
print(f"\n  {'Train':>12s} {'Test':>12s} {'F1':>8s} {'P':>8s} {'R':>8s} {'θ*':>6s}")
print(f"  {'-'*12} {'-'*12} {'-'*8} {'-'*8} {'-'*8} {'-'*6}")

transfer_results = {}

for train_lang in languages:
    train_docs = all_segmented[train_lang]
    train_ids = list(train_docs.keys())
    tp = compute_tp(train_ids, train_docs)

    for test_lang in languages:
        test_docs = all_segmented[test_lang]
        test_ids = list(test_docs.keys())
        metrics = find_best_threshold(test_ids, test_docs, tp)

        key = f"{train_lang}→{test_lang}"
        transfer_results[key] = metrics

        marker = "  ← same" if train_lang == test_lang else ""
        beat = ""
        if train_lang != test_lang:
            same_f1 = transfer_results.get(f"{test_lang}→{test_lang}", {}).get('f1', 0)
            if same_f1 > 0 and metrics['f1'] > same_f1:
                beat = "  ★ BEATS SAME-LANG"

        print(f"  {lang_names[train_lang]:>12s} {lang_names[test_lang]:>12s} "
              f"{metrics['f1']:>8.4f} {metrics['precision']:>8.4f} "
              f"{metrics['recall']:>8.4f} {metrics['threshold']:>6.2f}{marker}{beat}")

# ── 6. Zero-shot transfer (source threshold, no tuning) ──

print(f"\n{'='*70}")
print(f"  ZERO-SHOT TRANSFER (source language θ, no tuning on test)")
print(f"{'='*70}")
print(f"\n  {'Train':>12s} {'Test':>12s} {'F1':>8s} {'P':>8s} {'R':>8s} {'θ(src)':>8s}")
print(f"  {'-'*12} {'-'*12} {'-'*8} {'-'*8} {'-'*8} {'-'*8}")

for train_lang in languages:
    src_theta = transfer_results[f"{train_lang}→{train_lang}"]['threshold']
    train_docs = all_segmented[train_lang]
    tp = compute_tp(list(train_docs.keys()), train_docs)

    for test_lang in languages:
        if test_lang == train_lang:
            continue
        test_docs = all_segmented[test_lang]
        f1, p, r = evaluate_tp(list(test_docs.keys()), test_docs, tp, src_theta)

        print(f"  {lang_names[train_lang]:>12s} {lang_names[test_lang]:>12s} "
              f"{f1:>8.4f} {p:>8.4f} {r:>8.4f} {src_theta:>8.2f}")

# ── 7. Summary matrix ──

print(f"\n{'='*70}")
print(f"  TRANSFER MATRIX (F1)")
print(f"{'='*70}")

header = f"  {'Train\\Test':>12s}"
for tl in languages:
    header += f" {tl.upper():>8s}"
print(header)
print(f"  {'-'*12}" + f" {'-'*8}" * len(languages))

for trl in languages:
    row = f"  {trl.upper():>12s}"
    for tl in languages:
        key = f"{trl}→{tl}"
        f1 = transfer_results[key]['f1']
        row += f" {f1:>8.4f}"
    print(row)

# ── 8. Key findings ──

print(f"\n{'='*70}")
print(f"  KEY FINDINGS")
print(f"{'='*70}")

# Mesopotamian → Hittite
for src in ['akk', 'sux', 'elx']:
    key = f"{src}→hit"
    hit_same = transfer_results['hit→hit']['f1']
    cross = transfer_results[key]['f1']
    delta = cross - hit_same
    print(f"  {lang_names[src]:>12s} → Hittite: F1={cross:.4f} "
          f"(vs HIT→HIT {hit_same:.4f}, Δ={delta:+.4f})")

# Hittite → Mesopotamian
print()
for tgt in ['akk', 'sux', 'elx']:
    key = f"hit→{tgt}"
    same_key = f"{tgt}→{tgt}"
    same_f1 = transfer_results[same_key]['f1']
    cross = transfer_results[key]['f1']
    delta = cross - same_f1
    print(f"  Hittite → {lang_names[tgt]:>12s}: F1={cross:.4f} "
          f"(vs same-lang {same_f1:.4f}, Δ={delta:+.4f})")

Loading Hittite data...
  Raw tokens: 170,496
  Texts: 7,099
  Segmented documents: 5,909
  Words converted: 169,724/170,496 (99.5%)
  Total signs: 521,336
  Unique Unicode signs: 360

  Added 'hit' to doc_corpora: 7099 docs
  AKK: 13597 segmented docs
  SUX: 394 segmented docs
  ELX: 84 segmented docs
  HIT: 5909 segmented docs

  CROSS-LANGUAGE WORD BOUNDARY TRANSFER (4 languages)

         Train         Test       F1        P        R     θ*
  ------------ ------------ -------- -------- -------- ------
      Akkadian     Akkadian   0.9712   0.9463   0.9975   0.70  ← same
      Akkadian     Sumerian   0.9680   0.9382   0.9998   0.75
      Akkadian      Elamite   0.9949   0.9903   0.9995   0.35
      Akkadian      Hittite   0.5012   0.3541   0.8573   0.02
      Sumerian     Akkadian   0.9656   0.9363   0.9968   0.30
      Sumerian     Sumerian   0.9716   0.9455   0.9992   0.60  ← same
      Sumerian      Elamite   0.9949   0.9898   1.0000   0.33
      Sumerian      Hittite   0.5025   

In [10]:
# ============================================================
# CROSS-LANGUAGE WORD BOUNDARY TRANSFER
# ============================================================
# Train TP statistics on one cuneiform language,
# test on another. If it works, word boundaries are
# a property of the SCRIPT, not the LANGUAGE.
#
# Prerequisites: cells 1-13 (data loading)
# ============================================================

!pip install git+https://github.com/ancient-world-citation-analysis/cunei-tools.git

from cunei_tools import CuneiSeg
import numpy as np

# Collect documents per language
all_docs = {}
for lang in ['akk', 'sux', 'elx']:
    docs = list(doc_corpora[lang]['unicode'].values())
    docs = [d for d in docs if d and len(d.split()) > 2]
    all_docs[lang] = docs
    print(f"{lang.upper()}: {len(docs)} documents")

# ============================================================
# Experiment 1: Full cross-language transfer
# Train on language A, test on language B
# ============================================================
print(f"\n{'='*60}")
print(f"  CROSS-LANGUAGE WORD BOUNDARY TRANSFER")
print(f"{'='*60}")

lang_names = {'akk': 'Akkadian', 'sux': 'Sumerian', 'elx': 'Elamite'}

print(f"\n  {'Train →':>12s} {'Test →':>12s} {'F1':>8s} {'P':>8s} {'R':>8s} {'θ*':>6s}")
print(f"  {'-'*12} {'-'*12} {'-'*8} {'-'*8} {'-'*8} {'-'*6}")

transfer_results = {}

for train_lang in ['akk', 'sux', 'elx']:
    for test_lang in ['akk', 'sux', 'elx']:
        seg = CuneiSeg()
        seg.train(all_docs[train_lang])
        metrics = seg.find_optimal_threshold(all_docs[test_lang])

        key = f"{train_lang}→{test_lang}"
        transfer_results[key] = metrics

        marker = "  (same)" if train_lang == test_lang else ""
        print(f"  {lang_names[train_lang]:>12s} {lang_names[test_lang]:>12s} "
              f"{metrics['f1']:>8.4f} {metrics['precision']:>8.4f} "
              f"{metrics['recall']:>8.4f} {metrics['threshold']:>6.2f}{marker}")

# ============================================================
# Experiment 2: Transfer with FIXED threshold
# Train on A with A's optimal threshold, apply directly to B
# No threshold tuning on the test language
# ============================================================
print(f"\n{'='*60}")
print(f"  ZERO-SHOT TRANSFER (fixed threshold)")
print(f"{'='*60}")

# First get optimal thresholds per language
optimal_thresholds = {}
for lang in ['akk', 'sux', 'elx']:
    seg = CuneiSeg()
    seg.train(all_docs[lang])
    m = seg.find_optimal_threshold(all_docs[lang])
    optimal_thresholds[lang] = m['threshold']
    print(f"  {lang.upper()} optimal θ = {m['threshold']:.2f}")

print(f"\n  {'Train →':>12s} {'Test →':>12s} {'θ (from train)':>14s} {'F1':>8s}")
print(f"  {'-'*12} {'-'*12} {'-'*14} {'-'*8}")

for train_lang in ['akk', 'sux', 'elx']:
    for test_lang in ['akk', 'sux', 'elx']:
        if train_lang == test_lang:
            continue

        seg = CuneiSeg()
        seg.train(all_docs[train_lang])

        # Use train language's threshold, no tuning on test
        theta = optimal_thresholds[train_lang]

        # Evaluate manually with fixed threshold
        tp_total, fp_total, fn_total = 0, 0, 0
        for doc in all_docs[test_lang]:
            words = doc.split()
            continuous = doc.replace(' ', '')
            if len(continuous) < 3 or len(words) < 2:
                continue

            # Gold boundaries
            gold = set()
            pos = 0
            for w in words:
                pos += len(w)
                gold.add(pos)
            gold.discard(len(continuous))  # remove end

            # Predicted boundaries
            pred = set()
            for i in range(1, len(continuous)):
                bigram = (continuous[i-1], continuous[i])
                uni = seg.unigrams.get(continuous[i-1], 0)
                bi = seg.bigrams.get(bigram, 0)
                if uni > 0:
                    tp_val = bi / uni
                    if tp_val < theta:
                        pred.add(i)

            tp_total += len(gold & pred)
            fp_total += len(pred - gold)
            fn_total += len(gold - pred)

        p = tp_total / (tp_total + fp_total) if (tp_total + fp_total) else 0
        r = tp_total / (tp_total + fn_total) if (tp_total + fn_total) else 0
        f1 = 2 * p * r / (p + r) if (p + r) else 0

        print(f"  {lang_names[train_lang]:>12s} {lang_names[test_lang]:>12s} "
              f"{theta:>14.2f} {f1:>8.4f}")

# ============================================================
# Summary
# ============================================================
print(f"\n{'='*60}")
print(f"  SUMMARY")
print(f"{'='*60}")

print(f"\n  Same-language (baseline):")
for lang in ['akk', 'sux', 'elx']:
    key = f"{lang}→{lang}"
    print(f"    {lang.upper()}: F1 = {transfer_results[key]['f1']:.4f}")

print(f"\n  Cross-language transfer (with threshold tuning on test):")
for train_lang in ['akk', 'sux', 'elx']:
    for test_lang in ['akk', 'sux', 'elx']:
        if train_lang == test_lang:
            continue
        key = f"{train_lang}→{test_lang}"
        same_key = f"{test_lang}→{test_lang}"
        drop = transfer_results[same_key]['f1'] - transfer_results[key]['f1']
        print(f"    {train_lang.upper()}→{test_lang.upper()}: "
              f"F1 = {transfer_results[key]['f1']:.4f} "
              f"(Δ = {-drop:+.4f} vs same-lang)")

  Cloning https://github.com/ancient-world-citation-analysis/cunei-tools.git to /tmp/pip-req-build-84sl_jp4
  Running command git clone --filter=blob:none --quiet https://github.com/ancient-world-citation-analysis/cunei-tools.git /tmp/pip-req-build-84sl_jp4
  Resolved https://github.com/ancient-world-citation-analysis/cunei-tools.git to commit e3f2f56a8f914b6fa67d0349905a1ea4970c4fae
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for cunei-tools: filename=cunei_tools-0.1.0-py3-none-any.whl size=17898 sha256=2409f049ffbe643b88cf22f51c51e14c5dc115590076d451e556747585e2842b
  Stored in directory: /tmp/pip-ephem-wheel-cache-8s5vyaxy/wheels/ed/2f/05/5d2a91ebd894e60b49c24841c4f015784eed004e4b1b92ab95
Successfully built cunei-tools
AKK: 13502 documents
SUX: 394 documents
ELX: 84 documents

  CROSS-LANGUAGE WORD BOUNDARY TRANSFER

       Train →       Test →       F1        P        R     θ*

## Experiment 1: Embedding Comparison

Train fastText on Latin (char n-gram 2–5) vs Unicode (char n-gram 1–5).  
Evaluate: silhouette score, per-class silhouette, morpheme coherence.

In [9]:
# ============================================================
# EXPERIMENT 1: FASTTEXT EMBEDDINGS (cosine silhouette + pairwise coherence)
# ============================================================
from gensim.models import FastText
from sklearn.metrics import silhouette_samples

DIM = 100; WIN = 5; EPOCHS = 50
exp1_results = {}
ft_models = {}

for lang, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  EXP 1 — {lang.upper()}")
    print(f"{'='*60}")

    docs_latin = doc_corpora[lang]['latin']
    docs_unicode = doc_corpora[lang]['unicode']

    corpus_latin = [str(w).split() for w in df['form_latin'].dropna() if str(w).strip()]
    corpus_latin += [d.split() for d in docs_latin.values() if d]
    corpus_unicode = [str(w).split() for w in df['form_unicode'].dropna() if str(w).strip()]
    corpus_unicode += [d.split() for d in docs_unicode.values() if d]

    print(f"  Latin corpus: {len(corpus_latin)} sents, {sum(len(s) for s in corpus_latin)} tokens")
    print(f"  Unicode corpus: {len(corpus_unicode)} sents, {sum(len(s) for s in corpus_unicode)} tokens")

    print("  Training Latin fastText (char 2-5)...")
    model_l = FastText(sentences=corpus_latin, vector_size=DIM, window=WIN,
                       min_count=1, epochs=EPOCHS, seed=SEED, min_n=2, max_n=5, sg=1)
    print("  Training Unicode fastText (char 1-5)...")
    model_u = FastText(sentences=corpus_unicode, vector_size=DIM, window=WIN,
                       min_count=1, epochs=EPOCHS, seed=SEED, min_n=1, max_n=5, sg=1)

    ft_models[lang] = {'latin': model_l, 'unicode': model_u}
    results = {'lang': lang}

    # Silhouette (cosine metric, matching original)
    for name, model, repr_col in [('latin', model_l, 'form_latin'),
                                   ('unicode', model_u, 'form_unicode')]:
        vectors, labels = [], []
        for _, row in df.iterrows():
            word = str(row[repr_col])
            if word and word != 'nan' and word in model.wv:
                vectors.append(model.wv[word]); labels.append(row['pos_unified'])
        if len(vectors) < 50: continue
        vectors, labels = np.array(vectors), np.array(labels)
        if len(vectors) > 5000:
            idx = np.random.choice(len(vectors), 5000, replace=False)
            vectors, labels = vectors[idx], labels[idx]

        sil = silhouette_score(vectors, labels, metric='cosine')
        results[f'silhouette_{name}'] = sil

        sil_samples = silhouette_samples(vectors, labels, metric='cosine')
        per_class = {}
        for cls in np.unique(labels):
            mask = labels == cls
            if mask.sum() >= 5:
                per_class[cls] = float(np.mean(sil_samples[mask]))
        results[f'per_class_silhouette_{name}'] = per_class
        print(f"  {name} silhouette (cosine): {sil:.4f}")

        # Per-class detail
        for cls in sorted(per_class.keys()):
            print(f"    {cls:8s}: {per_class[cls]:.4f}")

    # Morpheme coherence (pairwise cosine, matching original)
    for name, model, repr_col in [('latin', model_l, 'form_latin'),
                                   ('unicode', model_u, 'form_unicode')]:
        group_col = 'morpheme_1' if 'morpheme_1' in df.columns else 'lemma'
        morph_data = df[df[group_col].notna()].copy()

        coh_scores = []
        for morph, group in morph_data.groupby(group_col):
            words = group[repr_col].dropna().astype(str).tolist()
            vecs = [model.wv[w] for w in words if w in model.wv and w != 'nan']
            if len(vecs) < 3: continue
            vecs = np.array(vecs)
            sim = cosine_similarity(vecs)
            n = len(vecs)
            coh_scores.append((sim.sum() - n) / (n * (n - 1)))

        coh = np.mean(coh_scores) if coh_scores else 0
        results[f'morpheme_coherence_{name}'] = coh
        print(f"  {name} morpheme coherence: {coh:.4f} ({len(coh_scores)} groups)")

    exp1_results[lang] = results



  EXP 1 — AKK
  Latin corpus: 1269389 sents, 2511338 tokens
  Unicode corpus: 1220431 sents, 4883746 tokens
  Training Latin fastText (char 2-5)...
  Training Unicode fastText (char 1-5)...
  latin silhouette (cosine): -0.1003
    ADJ     : -0.1570
    ADP     : 0.0233
    ADV     : -0.1012
    CN      : 0.3953
    CONJ    : 0.2854
    DET     : 0.1058
    DN      : -0.0169
    GN      : -0.0935
    INTJ    : -0.0545
    LN      : 0.3365
    MN      : -0.0077
    MOD     : 0.0701
    NOUN    : -0.1973
    NUM     : 0.0748
    PN      : -0.1488
    REL     : -0.1234
    RN      : -0.0548
    TN      : -0.0393
    VERB    : -0.1565
    X       : -0.1958
  unicode silhouette (cosine): -0.0974
    ADJ     : -0.1629
    ADP     : -0.0273
    ADV     : -0.1320
    CN      : -0.0105
    CONJ    : 0.2027
    DET     : 0.0292
    DN      : -0.1965
    GN      : -0.0764
    INTJ    : 0.0326
    LN      : -0.1317
    MN      : -0.0994
    MOD     : 0.0765
    NOUN    : -0.1983
    NUM     : 0.25

### Experiment 1 Extension: Cosine Similarity Graph Analysis

Build k-NN graphs from embeddings, measure POS-homophily, cross-representation edge agreement.

In [10]:
# ============================================================
# EXPERIMENT 1 EXTENSION: GRAPH ANALYSIS
# ============================================================
exp1_graph_results = {}

for lang, df in datasets.items():
    print(f"\n--- Graph Analysis: {lang.upper()} ---")
    model_l = ft_models[lang]['latin']
    model_u = ft_models[lang]['unicode']
    results = {'lang': lang}

    for repr_name, model, repr_col in [('latin', model_l, 'form_latin'),
                                        ('unicode', model_u, 'form_unicode')]:
        words, vectors, labels = [], [], []
        for _, row in df.iterrows():
            word = str(row[repr_col])
            if word and word != 'nan' and word in model.wv:
                words.append(word); vectors.append(model.wv[word]); labels.append(row['pos_unified'])
        if len(vectors) < 50: continue
        if len(vectors) > 2000:
            idx = np.random.choice(len(vectors), 2000, replace=False)
            words = [words[i] for i in idx]; vectors = [vectors[i] for i in idx]; labels = [labels[i] for i in idx]

        V = np.array(vectors)
        sim_matrix = cosine_similarity(V)
        top_k, threshold = 10, 0.5
        edges = set()
        for i in range(len(V)):
            top_indices = np.argsort(sim_matrix[i])[-top_k-1:-1][::-1]
            for j in top_indices:
                if sim_matrix[i][j] >= threshold:
                    edges.add((min(i,j), max(i,j)))

        same_pos = sum(1 for i,j in edges if labels[i]==labels[j])
        agreement = same_pos/len(edges) if edges else 0
        results[f'edges_{repr_name}'] = len(edges)
        results[f'pos_agreement_{repr_name}'] = agreement

        pos_pairs = Counter(tuple(sorted([labels[i],labels[j]])) for i,j in edges)
        results[f'top_pairs_{repr_name}'] = dict(pos_pairs.most_common(10))
        print(f"  {repr_name}: {len(edges)} edges, POS agreement: {agreement:.3f}")

    exp1_graph_results[lang] = results


--- Graph Analysis: AKK ---
  latin: 11811 edges, POS agreement: 0.648
  unicode: 14597 edges, POS agreement: 0.504

--- Graph Analysis: SUX ---
  latin: 11278 edges, POS agreement: 0.743
  unicode: 13849 edges, POS agreement: 0.637

--- Graph Analysis: ELX ---
  latin: 14207 edges, POS agreement: 0.463
  unicode: 14773 edges, POS agreement: 0.453


## Experiment 2: Word Boundary Inference

Transitional probability on unsegmented Unicode sign streams.  
Gold boundaries from known word-segmented documents.

In [11]:
# ============================================================
# EXPERIMENT 2: WORD BOUNDARY INFERENCE
# ============================================================
exp2_results = {}

for lang, df in datasets.items():
    print(f"\n--- Word Boundaries: {lang.upper()} ---")
    docs_unicode = doc_corpora[lang]['unicode']

    gold_data = []
    for doc in docs_unicode.values():
        if doc and len(doc.split()) > 2:
            boundaries, pos = [], 0
            for ch in doc:
                if ch == ' ': boundaries.append(pos)
                else: pos += 1
            continuous = doc.replace(' ', '')
            if len(continuous) > 3 and boundaries:
                gold_data.append({'segmented': doc, 'continuous': continuous, 'boundaries': boundaries})

    if len(gold_data) < 5:
        print(f"  Insufficient docs ({len(gold_data)})")
        exp2_results[lang] = {'lang': lang, 'note': 'insufficient_data'}
        continue

    print(f"  Gold documents: {len(gold_data)}")
    all_cont = ''.join(d['continuous'] for d in gold_data)
    unigrams = Counter(all_cont)
    bigrams = Counter(all_cont[i:i+2] for i in range(len(all_cont)-1))

    def tp(c1, c2):
        return bigrams[c1+c2] / unigrams[c1] if unigrams[c1] > 0 else 0

    best_f1, best_thresh = 0, 0
    for thresh in np.arange(0.05, 0.95, 0.05):
        tp_t, fp_t, fn_t = 0, 0, 0
        for d in gold_data:
            text = d['continuous']; gold_b = set(d['boundaries'])
            pred_b = {i for i in range(1, len(text)-1) if tp(text[i-1], text[i]) < thresh}
            tp_t += len(gold_b & pred_b); fp_t += len(pred_b - gold_b); fn_t += len(gold_b - pred_b)
        p = tp_t/(tp_t+fp_t) if (tp_t+fp_t) else 0
        r = tp_t/(tp_t+fn_t) if (tp_t+fn_t) else 0
        f1 = 2*p*r/(p+r) if (p+r) else 0
        if f1 > best_f1: best_f1, best_thresh = f1, thresh

    # Final metrics at best threshold
    tp_t, fp_t, fn_t = 0, 0, 0
    for d in gold_data:
        text = d['continuous']; gold_b = set(d['boundaries'])
        pred_b = {i for i in range(1, len(text)-1) if tp(text[i-1], text[i]) < best_thresh}
        tp_t += len(gold_b & pred_b); fp_t += len(pred_b - gold_b); fn_t += len(gold_b - pred_b)
    precision = tp_t/(tp_t+fp_t) if (tp_t+fp_t) else 0
    recall = tp_t/(tp_t+fn_t) if (tp_t+fn_t) else 0

    exp2_results[lang] = {
        'lang': lang, 'tp_f1': best_f1, 'tp_precision': precision,
        'tp_recall': recall, 'tp_threshold': best_thresh, 'n_docs': len(gold_data)
    }
    print(f"  TP: F1={best_f1:.4f} P={precision:.4f} R={recall:.4f} thresh={best_thresh:.2f}")


--- Word Boundaries: AKK ---
  Gold documents: 13419
  TP: F1=0.9687 P=0.9462 R=0.9922 thresh=0.70

--- Word Boundaries: SUX ---
  Gold documents: 394
  TP: F1=0.9712 P=0.9456 R=0.9982 thresh=0.60

--- Word Boundaries: ELX ---
  Gold documents: 84
  TP: F1=0.9733 P=0.9925 R=0.9547 thresh=0.90


In [12]:
# Install cunei-tools from GitHub
!pip install git+https://github.com/ancient-world-citation-analysis/cunei-tools.git

from cunei_tools import CuneiSeg
from sklearn.model_selection import KFold
import numpy as np

for lang in ['akk', 'sux', 'elx']:
    docs = list(doc_corpora[lang]['unicode'].values())
    docs = [d for d in docs if d and len(d.split()) > 2]

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    fold_f1s = []

    for train_idx, test_idx in kf.split(docs):
        train_docs = [docs[i] for i in train_idx]
        test_docs = [docs[i] for i in test_idx]

        seg = CuneiSeg(lang=lang)
        seg.train(train_docs)
        metrics = seg.find_optimal_threshold(test_docs)
        fold_f1s.append(metrics['f1'])

    print(f"{lang.upper()}: held-out F1 = {np.mean(fold_f1s):.4f} (±{np.std(fold_f1s):.4f})")

  Cloning https://github.com/ancient-world-citation-analysis/cunei-tools.git to /tmp/pip-req-build-rk_wqu5o
  Running command git clone --filter=blob:none --quiet https://github.com/ancient-world-citation-analysis/cunei-tools.git /tmp/pip-req-build-rk_wqu5o
  Resolved https://github.com/ancient-world-citation-analysis/cunei-tools.git to commit e3f2f56a8f914b6fa67d0349905a1ea4970c4fae
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for cunei-tools: filename=cunei_tools-0.1.0-py3-none-any.whl size=17898 sha256=6372af4de822941539e79f34cb75c6f3af9a863fe9fd0ac333cad6af8dd75939
  Stored in directory: /tmp/pip-ephem-wheel-cache-fpk20m94/wheels/ed/2f/05/5d2a91ebd894e60b49c24841c4f015784eed004e4b1b92ab95
Successfully built cunei-tools
AKK: held-out F1 = 0.9711 (±0.0015)
SUX: held-out F1 = 0.9715 (±0.0025)
ELX: held-out F1 = 0.9886 (±0.0054)


In [12]:
# Export threshold sweep data for figures
import csv

for lang in ['akk', 'sux', 'elx']:
    docs_unicode = doc_corpora[lang]['unicode']
    gold_data = []
    for doc in docs_unicode.values():
        if doc and len(doc.split()) > 2:
            boundaries, pos = [], 0
            for ch in doc:
                if ch == ' ': boundaries.append(pos)
                else: pos += 1
            continuous = doc.replace(' ', '')
            if len(continuous) > 3 and boundaries:
                gold_data.append({'continuous': continuous, 'boundaries': boundaries})

    all_cont = ''.join(d['continuous'] for d in gold_data)
    unigrams = Counter(all_cont)
    bigrams = Counter(all_cont[i:i+2] for i in range(len(all_cont)-1))

    rows = []
    for thresh in np.arange(0.05, 0.96, 0.05):
        tp_t, fp_t, fn_t = 0, 0, 0
        for d in gold_data:
            text = d['continuous']; gold_b = set(d['boundaries'])
            pred_b = {i for i in range(1, len(text)-1)
                      if bigrams[text[i-1]+text[i]] / unigrams[text[i-1]] < thresh
                      if unigrams[text[i-1]] > 0}
            tp_t += len(gold_b & pred_b)
            fp_t += len(pred_b - gold_b)
            fn_t += len(gold_b - pred_b)
        p = tp_t/(tp_t+fp_t) if (tp_t+fp_t) else 0
        r = tp_t/(tp_t+fn_t) if (tp_t+fn_t) else 0
        f1 = 2*p*r/(p+r) if (p+r) else 0
        rows.append({'threshold': thresh, 'f1': f1, 'precision': p, 'recall': r})

    pd.DataFrame(rows).to_csv(f'threshold_sweep_{lang}.csv', index=False)
    print(f"{lang}: saved {len(rows)} threshold points")

akk: saved 19 threshold points
sux: saved 19 threshold points
elx: saved 19 threshold points


## Experiment 3: POS Classification

Three classifiers × two representations × two tagsets (unified + grammatical).  
Models: char n-gram Logistic Regression, k-NN on fastText, BiLSTM.

In [13]:

# ============================================================
# EXPERIMENT 3: POS CLASSIFICATION (LR + k-NN)
# ============================================================
exp3_results = {}
MIN_CLASS_COUNT = 20

for lang, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  EXP 3 — {lang.upper()}")
    print(f"{'='*60}")
    model_l = ft_models[lang]['latin']
    model_u = ft_models[lang]['unicode']
    results = {'lang': lang}

    for tagset_name, tag_col in [('unified', 'pos_unified'), ('grammatical', 'pos_grammatical')]:
        print(f"\n  Tagset: {tagset_name}")
        counts = df[tag_col].value_counts()
        valid = counts[counts >= MIN_CLASS_COUNT].index.tolist()
        clf_df = df[df[tag_col].isin(valid)].copy()
        if len(clf_df) > 50000:
            clf_df = clf_df.sample(50000, random_state=SEED)
            print(f"    Subsampled to {len(clf_df):,}")
        if len(valid) < 2 or len(clf_df) < 50:
            print(f"    Insufficient data"); continue
        print(f"    {len(valid)} classes, {len(clf_df):,} tokens")

        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

        # Model 1: Char n-gram LR
        for repr_name, repr_col in [('latin', 'form_latin'), ('unicode', 'form_unicode')]:
            texts = clf_df[repr_col].astype(str).values
            labels = clf_df[tag_col].values
            try:
                vec = CountVectorizer(analyzer='char', ngram_range=(1,4))
                X = vec.fit_transform(texts)
                clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED, C=1.0)
                scores = cross_val_score(clf, X, labels, cv=cv, scoring='f1_macro')
                results[f'ngram_lr_{tagset_name}_{repr_name}'] = scores.mean()
                print(f"    N-gram LR {repr_name}: {scores.mean():.4f} (±{scores.std():.4f})")
            except Exception as e:
                print(f"    N-gram LR {repr_name} failed: {e}")

        # Model 2: k-NN on fastText
        for repr_name, model, repr_col in [('latin', model_l, 'form_latin'),
                                            ('unicode', model_u, 'form_unicode')]:
            X, y = [], []
            for _, row in clf_df.iterrows():
                word = str(row[repr_col])
                if word and word != 'nan':
                    try: X.append(model.wv[word]); y.append(row[tag_col])
                    except KeyError: continue
            if len(X) < 50: continue
            X, y = np.array(X), np.array(y)
            knn = KNeighborsClassifier(n_neighbors=min(5, len(X)//5))
            try:
                scores = cross_val_score(knn, X, y, cv=cv, scoring='f1_macro')
                results[f'knn_{tagset_name}_{repr_name}'] = scores.mean()
                print(f"    k-NN {repr_name}: {scores.mean():.4f} (±{scores.std():.4f})")
            except Exception as e:
                print(f"    k-NN {repr_name} failed: {e}")

    exp3_results[lang] = results


  EXP 3 — AKK

  Tagset: unified
    Subsampled to 50,000
    24 classes, 50,000 tokens
    N-gram LR latin: 0.6926 (±0.0305)
    N-gram LR unicode: 0.6210 (±0.0211)
    k-NN latin: 0.7068 (±0.0180)
    k-NN unicode: 0.5836 (±0.0178)

  Tagset: grammatical
    Subsampled to 50,000
    14 classes, 50,000 tokens
    N-gram LR latin: 0.6653 (±0.0051)
    N-gram LR unicode: 0.6432 (±0.0045)
    k-NN latin: 0.7163 (±0.0264)
    k-NN unicode: 0.6467 (±0.0172)

  EXP 3 — SUX

  Tagset: unified
    Subsampled to 50,000
    14 classes, 50,000 tokens
    N-gram LR latin: 0.8472 (±0.0057)
    N-gram LR unicode: 0.7503 (±0.0119)
    k-NN latin: 0.8367 (±0.0112)
    k-NN unicode: 0.7602 (±0.0059)

  Tagset: grammatical
    Subsampled to 50,000
    7 classes, 50,000 tokens
    N-gram LR latin: 0.8805 (±0.0038)
    N-gram LR unicode: 0.8225 (±0.0064)
    k-NN latin: 0.8565 (±0.0096)
    k-NN unicode: 0.8141 (±0.0147)

  EXP 3 — ELX

  Tagset: unified
    7 classes, 13,383 tokens
    N-gram LR latin:

In [14]:
# ============================================================
# EXPERIMENT 3: BiLSTM (memory-optimized)
# ============================================================
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

exp3_bilstm_results = {}
EMBED_DIM = 64; HIDDEN_DIM = 128; N_LAYERS = 2; DROPOUT = 0.3
LR = 0.001; BATCH_SIZE = 64; N_EPOCHS = 30

for lang, df in datasets.items():
    print(f"\n--- BiLSTM: {lang.upper()} ---")
    torch.cuda.empty_cache()

    for pos_col in ['pos_unified', 'pos_grammatical']:
        counts = df[pos_col].value_counts()
        valid = counts[counts >= MIN_CLASS_COUNT].index.tolist()
        clf_df = df[df[pos_col].isin(valid)].copy()
        if len(valid) < 2: continue

        if len(clf_df) > 20000:
            clf_df = clf_df.sample(20000, random_state=SEED)
            print(f"  Subsampled to {len(clf_df):,}")

        le = LabelEncoder()
        labels_enc = le.fit_transform(clf_df[pos_col])
        n_classes = len(le.classes_)

        for repr_name, repr_col in [('latin', 'form_latin'), ('unicode', 'form_unicode')]:
            texts = clf_df[repr_col].astype(str).tolist()
            char2idx = {'<pad>': 0, '<unk>': 1}
            for t in texts:
                for ch in t:
                    if ch not in char2idx: char2idx[ch] = len(char2idx)

            max_len = min(max(len(t) for t in texts), 100)
            X = np.zeros((len(texts), max_len), dtype=np.int64)
            for i, t in enumerate(texts):
                for j, ch in enumerate(t[:max_len]):
                    X[i, j] = char2idx.get(ch, 1)

            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
            fold_scores = []
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

            for fold, (train_idx, val_idx) in enumerate(cv.split(X, labels_enc)):
                X_tr = torch.LongTensor(X[train_idx]).to(device)
                y_tr = torch.LongTensor(labels_enc[train_idx]).to(device)
                X_va = torch.LongTensor(X[val_idx]).to(device)
                y_va = labels_enc[val_idx]

                class BiLSTM(nn.Module):
                    def __init__(self):
                        super().__init__()
                        self.embed = nn.Embedding(len(char2idx), EMBED_DIM, padding_idx=0)
                        self.lstm = nn.LSTM(EMBED_DIM, HIDDEN_DIM, N_LAYERS,
                                           bidirectional=True, batch_first=True, dropout=DROPOUT)
                        self.fc = nn.Linear(HIDDEN_DIM*2, n_classes)
                        self.drop = nn.Dropout(DROPOUT)
                    def forward(self, x):
                        emb = self.drop(self.embed(x))
                        _, (h, _) = self.lstm(emb)
                        h = torch.cat([h[-2], h[-1]], dim=1)
                        return self.fc(self.drop(h))

                model = BiLSTM().to(device)
                opt = torch.optim.Adam(model.parameters(), lr=LR)
                wts = torch.FloatTensor([1.0/max((labels_enc[train_idx]==c).sum(),1) for c in range(n_classes)]).to(device)
                crit = nn.CrossEntropyLoss(weight=wts)

                loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
                model.train()
                for ep in range(N_EPOCHS):
                    for xb, yb in loader:
                        opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()

                model.eval()
                with torch.no_grad():
                    all_preds = []
                    for k in range(0, len(X_va), 256):
                        all_preds.append(model(X_va[k:k+256]).argmax(dim=1).cpu().numpy())
                    preds = np.concatenate(all_preds)
                fold_scores.append(f1_score(y_va, preds, average='macro'))

                del model, X_tr, y_tr, X_va
                torch.cuda.empty_cache()

            key = f'bilstm_{pos_col}_{repr_name}'
            exp3_bilstm_results.setdefault(lang, {})[key] = np.mean(fold_scores)
            print(f"  {pos_col} {repr_name}: {np.mean(fold_scores):.4f} (±{np.std(fold_scores):.4f})")


--- BiLSTM: AKK ---
  Subsampled to 20,000
  pos_unified latin: 0.6226 (±0.0253)
  pos_unified unicode: 0.4956 (±0.0164)
  Subsampled to 20,000
  pos_grammatical latin: 0.6425 (±0.0111)
  pos_grammatical unicode: 0.5643 (±0.0062)

--- BiLSTM: SUX ---
  Subsampled to 20,000
  pos_unified latin: 0.7602 (±0.0225)
  pos_unified unicode: 0.6456 (±0.0201)
  Subsampled to 20,000
  pos_grammatical latin: 0.8515 (±0.0204)
  pos_grammatical unicode: 0.7682 (±0.0117)

--- BiLSTM: ELX ---
  pos_unified latin: 0.6509 (±0.0080)
  pos_unified unicode: 0.6354 (±0.0094)
  pos_grammatical latin: 0.6239 (±0.0070)
  pos_grammatical unicode: 0.6065 (±0.0098)


## Experiment 3b: Separated POS vs NER Analysis

Three sub-tasks to test whether Unicode helps specifically for entity recognition:
1. Grammatical POS only (entities excluded)
2. Entity detection (binary)
3. Entity typing (PN/DN/GN, entities only)


In [15]:
# ============================================================
# EXPERIMENT 3b: SEPARATED POS vs NER ANALYSIS
# ============================================================
exp3b_results = {}

for lang, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  EXP 3b — {lang.upper()}: Separated POS vs NER")
    print(f"{'='*60}")
    model_l = ft_models[lang]['latin']
    model_u = ft_models[lang]['unicode']
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    results = {'lang': lang}

    # ── Task 1: Grammatical POS only (exclude entities) ──
    gram_df = df[df['ner_tag'] == 'O'].copy()
    counts = gram_df['pos_grammatical'].value_counts()
    valid = counts[counts >= MIN_CLASS_COUNT].index.tolist()
    gram_clf = gram_df[gram_df['pos_grammatical'].isin(valid)]
    if len(gram_clf) > 50000:
        gram_clf = gram_clf.sample(50000, random_state=SEED)

    if len(valid) >= 2:
        print(f"\n  Task 1: Grammatical POS (entities excluded)")
        print(f"    {len(valid)} classes, {len(gram_clf):,} tokens")
        for repr_name, repr_col in [('latin', 'form_latin'), ('unicode', 'form_unicode')]:
            texts = gram_clf[repr_col].astype(str).values
            labels = gram_clf['pos_grammatical'].values
            vec = CountVectorizer(analyzer='char', ngram_range=(1,4))
            X = vec.fit_transform(texts)
            clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
            scores = cross_val_score(clf, X, labels, cv=cv, scoring='f1_macro')
            results[f'gram_only_{repr_name}'] = scores.mean()
            print(f"    N-gram LR {repr_name}: {scores.mean():.4f} (±{scores.std():.4f})")

    # ── Task 2: Entity detection (binary) ──
    detect_df = df.copy()
    detect_df['is_entity'] = (detect_df['ner_tag'] != 'O').astype(str)
    if len(detect_df) > 50000:
        detect_df = detect_df.sample(50000, random_state=SEED)

    print(f"\n  Task 2: Entity detection (binary)")
    print(f"    {len(detect_df):,} tokens, {(detect_df['is_entity']=='True').mean():.1%} entities")
    for repr_name, repr_col in [('latin', 'form_latin'), ('unicode', 'form_unicode')]:
        texts = detect_df[repr_col].astype(str).values
        labels = detect_df['is_entity'].values
        vec = CountVectorizer(analyzer='char', ngram_range=(1,4))
        X = vec.fit_transform(texts)
        clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
        scores = cross_val_score(clf, X, labels, cv=cv, scoring='f1_macro')
        results[f'entity_detect_{repr_name}'] = scores.mean()
        print(f"    N-gram LR {repr_name}: {scores.mean():.4f} (±{scores.std():.4f})")

    # ── Task 3: Entity typing (PN/DN/GN/etc., entities only) ──
    ent_df = df[df['ner_tag'] != 'O'].copy()
    ent_counts = ent_df['ner_tag'].value_counts()
    ent_valid = ent_counts[ent_counts >= MIN_CLASS_COUNT].index.tolist()
    ent_clf = ent_df[ent_df['ner_tag'].isin(ent_valid)]
    if len(ent_clf) > 50000:
        ent_clf = ent_clf.sample(50000, random_state=SEED)

    if len(ent_valid) >= 2:
        print(f"\n  Task 3: Entity typing (entities only)")
        print(f"    {len(ent_valid)} types, {len(ent_clf):,} tokens")
        for repr_name, repr_col in [('latin', 'form_latin'), ('unicode', 'form_unicode')]:
            texts = ent_clf[repr_col].astype(str).values
            labels = ent_clf['ner_tag'].values
            vec = CountVectorizer(analyzer='char', ngram_range=(1,4))
            X = vec.fit_transform(texts)
            clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
            scores = cross_val_score(clf, X, labels, cv=cv, scoring='f1_macro')
            results[f'entity_type_{repr_name}'] = scores.mean()
            print(f"    N-gram LR {repr_name}: {scores.mean():.4f} (±{scores.std():.4f})")

    exp3b_results[lang] = results



  EXP 3b — AKK: Separated POS vs NER

  Task 1: Grammatical POS (entities excluded)
    13 classes, 50,000 tokens
    N-gram LR latin: 0.6688 (±0.0041)
    N-gram LR unicode: 0.6574 (±0.0058)

  Task 2: Entity detection (binary)
    50,000 tokens, 12.7% entities
    N-gram LR latin: 0.9365 (±0.0033)
    N-gram LR unicode: 0.8390 (±0.0041)

  Task 3: Entity typing (entities only)
    11 types, 50,000 tokens
    N-gram LR latin: 0.8970 (±0.0088)
    N-gram LR unicode: 0.8333 (±0.0060)

  EXP 3b — SUX: Separated POS vs NER

  Task 1: Grammatical POS (entities excluded)
    6 classes, 50,000 tokens
    N-gram LR latin: 0.8718 (±0.0099)
    N-gram LR unicode: 0.8324 (±0.0119)

  Task 2: Entity detection (binary)
    50,000 tokens, 9.1% entities
    N-gram LR latin: 0.9472 (±0.0034)
    N-gram LR unicode: 0.8958 (±0.0027)

  Task 3: Entity typing (entities only)
    8 types, 13,384 tokens
    N-gram LR latin: 0.9573 (±0.0039)
    N-gram LR unicode: 0.9225 (±0.0063)

  EXP 3b — ELX: Separate

## Experiment 3c: Concatenation Experiment

Test whether combining Latin + Unicode features improves over either alone.
Run on all tasks: unified POS, grammatical POS, entity detection, entity typing.


In [16]:
# ============================================================
# EXPERIMENT 3c: CONCATENATION (Latin + Unicode)
# ============================================================
exp3c_results = {}

for lang, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  EXP 3c — {lang.upper()}: Concatenation")
    print(f"{'='*60}")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    results = {'lang': lang}

    tasks = []

    # Task: Unified POS
    counts = df['pos_unified'].value_counts()
    valid = counts[counts >= MIN_CLASS_COUNT].index.tolist()
    task_df = df[df['pos_unified'].isin(valid)].copy()
    if len(task_df) > 50000:
        task_df = task_df.sample(50000, random_state=SEED)
    if len(valid) >= 2:
        tasks.append(('unified_pos', task_df, 'pos_unified'))

    # Task: Grammatical POS (no entities)
    gram_df = df[df['ner_tag'] == 'O'].copy()
    counts = gram_df['pos_grammatical'].value_counts()
    valid = counts[counts >= MIN_CLASS_COUNT].index.tolist()
    gram_clf = gram_df[gram_df['pos_grammatical'].isin(valid)]
    if len(gram_clf) > 50000:
        gram_clf = gram_clf.sample(50000, random_state=SEED)
    if len(valid) >= 2:
        tasks.append(('gram_only', gram_clf, 'pos_grammatical'))

    # Task: Entity detection
    det_df = df.copy()
    det_df['is_entity'] = (det_df['ner_tag'] != 'O').astype(str)
    if len(det_df) > 50000:
        det_df = det_df.sample(50000, random_state=SEED)
    tasks.append(('entity_detect', det_df, 'is_entity'))

    # Task: Entity typing
    ent_df = df[df['ner_tag'] != 'O'].copy()
    ent_counts = ent_df['ner_tag'].value_counts()
    ent_valid = ent_counts[ent_counts >= MIN_CLASS_COUNT].index.tolist()
    ent_clf = ent_df[ent_df['ner_tag'].isin(ent_valid)]
    if len(ent_clf) > 50000:
        ent_clf = ent_clf.sample(50000, random_state=SEED)
    if len(ent_valid) >= 2:
        tasks.append(('entity_type', ent_clf, 'ner_tag'))

    for task_name, task_df, label_col in tasks:
        print(f"\n  {task_name}: {len(task_df):,} tokens")
        texts_l = task_df['form_latin'].astype(str).values
        texts_u = task_df['form_unicode'].astype(str).values
        labels = task_df[label_col].values

        vec_l = CountVectorizer(analyzer='char', ngram_range=(1,4))
        vec_u = CountVectorizer(analyzer='char', ngram_range=(1,4))
        X_l = vec_l.fit_transform(texts_l)
        X_u = vec_u.fit_transform(texts_u)

        from scipy.sparse import hstack
        X_concat = hstack([X_l, X_u])

        clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)

        # Latin only
        s_l = cross_val_score(clf, X_l, labels, cv=cv, scoring='f1_macro')
        # Unicode only
        s_u = cross_val_score(clf, X_u, labels, cv=cv, scoring='f1_macro')
        # Concatenated
        s_c = cross_val_score(clf, X_concat, labels, cv=cv, scoring='f1_macro')

        results[f'{task_name}_latin'] = s_l.mean()
        results[f'{task_name}_unicode'] = s_u.mean()
        results[f'{task_name}_concat'] = s_c.mean()

        best = max(s_l.mean(), s_u.mean(), s_c.mean())
        print(f"    Latin:  {s_l.mean():.4f} (±{s_l.std():.4f})")
        print(f"    Unicode: {s_u.mean():.4f} (±{s_u.std():.4f})")
        print(f"    Concat:  {s_c.mean():.4f} (±{s_c.std():.4f}) {'*** BEST' if s_c.mean() == best else ''}")

    exp3c_results[lang] = results



  EXP 3c — AKK: Concatenation

  unified_pos: 50,000 tokens
    Latin:  0.6926 (±0.0305)
    Unicode: 0.6210 (±0.0211)
    Concat:  0.7124 (±0.0284) *** BEST

  gram_only: 50,000 tokens
    Latin:  0.6688 (±0.0041)
    Unicode: 0.6574 (±0.0058)
    Concat:  0.6816 (±0.0041) *** BEST

  entity_detect: 50,000 tokens
    Latin:  0.9365 (±0.0033)
    Unicode: 0.8390 (±0.0041)
    Concat:  0.9385 (±0.0018) *** BEST

  entity_type: 50,000 tokens
    Latin:  0.8970 (±0.0088)
    Unicode: 0.8333 (±0.0060)
    Concat:  0.9098 (±0.0075) *** BEST

  EXP 3c — SUX: Concatenation

  unified_pos: 50,000 tokens
    Latin:  0.8472 (±0.0057)
    Unicode: 0.7503 (±0.0119)
    Concat:  0.8701 (±0.0084) *** BEST

  gram_only: 50,000 tokens
    Latin:  0.8718 (±0.0099)
    Unicode: 0.8324 (±0.0119)
    Concat:  0.8876 (±0.0103) *** BEST

  entity_detect: 50,000 tokens
    Latin:  0.9472 (±0.0034)
    Unicode: 0.8958 (±0.0027)
    Concat:  0.9543 (±0.0025) *** BEST

  entity_type: 13,384 tokens
    Latin:  

In [26]:
# ============================================================
# STATISTICAL SIGNIFICANCE TESTS
# ============================================================
# Paired bootstrap test on all key Latin vs Unicode and
# Concat vs Best comparisons.
#
# Add this cell after Exp 3c in your notebook.
# ============================================================

from scipy.sparse import hstack
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

def paired_bootstrap_test(y_true, preds_a, preds_b, n_bootstrap=2000, seed=42):
    """
    Paired bootstrap significance test.
    Tests whether system A is significantly different from system B.

    Args:
        y_true: Gold labels
        preds_a: Predictions from system A
        preds_b: Predictions from system B
        n_bootstrap: Number of bootstrap samples
        seed: Random seed

    Returns:
        dict with p_value, score_a, score_b, delta, ci_lower, ci_upper
    """
    rng = np.random.RandomState(seed)
    n = len(y_true)

    score_a = f1_score(y_true, preds_a, average='macro', zero_division=0)
    score_b = f1_score(y_true, preds_b, average='macro', zero_division=0)
    observed_delta = score_a - score_b

    deltas = []
    count_b_wins = 0

    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, size=n)
        boot_true = y_true[idx]
        boot_a = preds_a[idx]
        boot_b = preds_b[idx]

        sa = f1_score(boot_true, boot_a, average='macro', zero_division=0)
        sb = f1_score(boot_true, boot_b, average='macro', zero_division=0)
        delta = sa - sb
        deltas.append(delta)

        # Two-sided: count how often the sign flips
        if observed_delta >= 0 and delta <= 0:
            count_b_wins += 1
        elif observed_delta < 0 and delta >= 0:
            count_b_wins += 1

    deltas = np.array(deltas)
    p_value = (2 * count_b_wins) / n_bootstrap  # two-sided
    p_value = min(p_value, 1.0)

    return {
        'score_a': score_a,
        'score_b': score_b,
        'delta': observed_delta,
        'p_value': p_value,
        'ci_lower': np.percentile(deltas, 2.5),
        'ci_upper': np.percentile(deltas, 97.5),
        'significant_005': p_value < 0.05,
        'significant_001': p_value < 0.01,
    }


def run_significance_for_task(texts_l, texts_u, labels, task_name, lang):
    """
    Train Latin, Unicode, and Concat models, collect predictions,
    run paired bootstrap on all pairs.
    """
    vec_l = CountVectorizer(analyzer='char', ngram_range=(1, 4))
    vec_u = CountVectorizer(analyzer='char', ngram_range=(1, 4))
    X_l = vec_l.fit_transform(texts_l)
    X_u = vec_u.fit_transform(texts_u)
    X_c = hstack([X_l, X_u])

    labels = np.array(labels)

    # Collect cross-validated predictions for each system
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    preds_l = np.empty(len(labels), dtype=labels.dtype)
    preds_u = np.empty(len(labels), dtype=labels.dtype)
    preds_c = np.empty(len(labels), dtype=labels.dtype)

    for train_idx, test_idx in cv.split(X_l, labels):
        clf_l = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED, C=1.0)
        clf_u = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED, C=1.0)
        clf_c = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED, C=1.0)

        clf_l.fit(X_l[train_idx], labels[train_idx])
        clf_u.fit(X_u[train_idx], labels[train_idx])
        clf_c.fit(X_c[train_idx], labels[train_idx])

        preds_l[test_idx] = clf_l.predict(X_l[test_idx])
        preds_u[test_idx] = clf_u.predict(X_u[test_idx])
        preds_c[test_idx] = clf_c.predict(X_c[test_idx])

    # Run bootstrap tests
    results = {}

    # Latin vs Unicode
    test_lu = paired_bootstrap_test(labels, preds_l, preds_u)
    sig_lu = '***' if test_lu['significant_001'] else ('**' if test_lu['significant_005'] else 'ns')
    results['latin_vs_unicode'] = test_lu

    # Concat vs Latin
    test_cl = paired_bootstrap_test(labels, preds_c, preds_l)
    sig_cl = '***' if test_cl['significant_001'] else ('**' if test_cl['significant_005'] else 'ns')
    results['concat_vs_latin'] = test_cl

    # Concat vs Unicode
    test_cu = paired_bootstrap_test(labels, preds_c, preds_u)
    sig_cu = '***' if test_cu['significant_001'] else ('**' if test_cu['significant_005'] else 'ns')
    results['concat_vs_unicode'] = test_cu

    print(f"\n  {task_name}:")
    print(f"    Latin={test_lu['score_a']:.4f}  Unicode={test_lu['score_b']:.4f}  "
          f"Concat={test_cl['score_a']:.4f}")
    print(f"    Latin vs Unicode:  Δ={test_lu['delta']:+.4f}  p={test_lu['p_value']:.4f}  "
          f"CI=[{test_lu['ci_lower']:+.4f}, {test_lu['ci_upper']:+.4f}]  {sig_lu}")
    print(f"    Concat vs Latin:   Δ={test_cl['delta']:+.4f}  p={test_cl['p_value']:.4f}  "
          f"CI=[{test_cl['ci_lower']:+.4f}, {test_cl['ci_upper']:+.4f}]  {sig_cl}")
    print(f"    Concat vs Unicode: Δ={test_cu['delta']:+.4f}  p={test_cu['p_value']:.4f}  "
          f"CI=[{test_cu['ci_lower']:+.4f}, {test_cu['ci_upper']:+.4f}]  {sig_cu}")

    return results


# ============================================================
# RUN ALL SIGNIFICANCE TESTS
# ============================================================
all_sig_results = {}

for lang, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  SIGNIFICANCE TESTS — {lang.upper()}")
    print(f"{'='*60}")

    lang_results = {}

    # ── Unified POS ──
    counts = df['pos_unified'].value_counts()
    valid = counts[counts >= MIN_CLASS_COUNT].index.tolist()
    task_df = df[df['pos_unified'].isin(valid)].copy()
    if len(task_df) > 10000:
        task_df = task_df.sample(10000, random_state=SEED)

    if len(valid) >= 2:
        lang_results['unified_pos'] = run_significance_for_task(
            task_df['form_latin'].astype(str).values,
            task_df['form_unicode'].astype(str).values,
            task_df['pos_unified'].values,
            'Unified POS', lang
        )

    # ── Grammatical POS (entities excluded) ──
    gram_df = df[df['ner_tag'] == 'O'].copy()
    counts = gram_df['pos_grammatical'].value_counts()
    valid = counts[counts >= MIN_CLASS_COUNT].index.tolist()
    gram_clf = gram_df[gram_df['pos_grammatical'].isin(valid)]
    if len(gram_clf) > 10000:
        gram_clf = gram_clf.sample(10000, random_state=SEED)

    if len(valid) >= 2:
        lang_results['gram_pos'] = run_significance_for_task(
            gram_clf['form_latin'].astype(str).values,
            gram_clf['form_unicode'].astype(str).values,
            gram_clf['pos_grammatical'].values,
            'Grammatical POS (no entities)', lang
        )

    # ── Entity detection (binary) ──
    det_df = df.copy()
    det_df['is_entity'] = (det_df['ner_tag'] != 'O').astype(str)
    if len(det_df) > 10000:
        det_df = det_df.sample(10000, random_state=SEED)

    lang_results['entity_detect'] = run_significance_for_task(
        det_df['form_latin'].astype(str).values,
        det_df['form_unicode'].astype(str).values,
        det_df['is_entity'].values,
        'Entity detection (binary)', lang
    )

    # ── Entity typing ──
    ent_df = df[df['ner_tag'] != 'O'].copy()
    ent_counts = ent_df['ner_tag'].value_counts()
    ent_valid = ent_counts[ent_counts >= MIN_CLASS_COUNT].index.tolist()
    ent_clf = ent_df[ent_df['ner_tag'].isin(ent_valid)]
    if len(ent_clf) > 10000:
        ent_clf = ent_clf.sample(10000, random_state=SEED)

    if len(ent_valid) >= 2:
        lang_results['entity_type'] = run_significance_for_task(
            ent_clf['form_latin'].astype(str).values,
            ent_clf['form_unicode'].astype(str).values,
            ent_clf['ner_tag'].values,
            'Entity typing', lang
        )

    all_sig_results[lang] = lang_results

# ============================================================
# SUMMARY TABLE
# ============================================================
print(f"\n{'='*60}")
print(f"  SIGNIFICANCE SUMMARY")
print(f"{'='*60}")
print(f"\n  *** = p < 0.01,  ** = p < 0.05,  ns = not significant\n")
print(f"  {'Comparison':<25s} {'Task':<20s} {'AKK':>8s} {'SUX':>8s} {'ELX':>8s}")
print(f"  {'-'*25} {'-'*20} {'-'*8} {'-'*8} {'-'*8}")

for task in ['unified_pos', 'gram_pos', 'entity_detect', 'entity_type']:
    for comp in ['latin_vs_unicode', 'concat_vs_latin', 'concat_vs_unicode']:
        row = []
        for lang in ['akk', 'sux', 'elx']:
            if lang in all_sig_results and task in all_sig_results[lang]:
                r = all_sig_results[lang][task].get(comp, {})
                if r:
                    p = r['p_value']
                    sig = '***' if p < 0.01 else ('**' if p < 0.05 else 'ns')
                    row.append(f"{r['delta']:+.3f}{sig}")
                else:
                    row.append('—')
            else:
                row.append('—')

        comp_label = comp.replace('_', ' ').title()
        task_label = task.replace('_', ' ').title()
        print(f"  {comp_label:<25s} {task_label:<20s} {row[0]:>8s} {row[1]:>8s} {row[2]:>8s}")
    print()



  SIGNIFICANCE TESTS — AKK

  Unified POS:
    Latin=0.6043  Unicode=0.5440  Concat=0.6284
    Latin vs Unicode:  Δ=+0.0603  p=0.0000  CI=[+0.0461, +0.0924]  ***
    Concat vs Latin:   Δ=+0.0240  p=0.0010  CI=[+0.0155, +0.0320]  ***
    Concat vs Unicode: Δ=+0.0843  p=0.0000  CI=[+0.0725, +0.1164]  ***

  Grammatical POS (no entities):
    Latin=0.5978  Unicode=0.6091  Concat=0.6181
    Latin vs Unicode:  Δ=-0.0113  p=0.0200  CI=[-0.0206, -0.0017]  **
    Concat vs Latin:   Δ=+0.0203  p=0.0000  CI=[+0.0157, +0.0248]  ***
    Concat vs Unicode: Δ=+0.0090  p=0.0410  CI=[+0.0004, +0.0177]  **

  Entity detection (binary):
    Latin=0.9207  Unicode=0.8120  Concat=0.9256
    Latin vs Unicode:  Δ=+0.1087  p=0.0000  CI=[+0.0980, +0.1197]  ***
    Concat vs Latin:   Δ=+0.0049  p=0.0050  CI=[+0.0017, +0.0085]  ***
    Concat vs Unicode: Δ=+0.1136  p=0.0000  CI=[+0.1034, +0.1240]  ***

  Entity typing:
    Latin=0.8578  Unicode=0.7600  Concat=0.8646
    Latin vs Unicode:  Δ=+0.0978  p=0.0000  C

## Experiment 4: Lemmatization

Character-level LSTM seq2seq: form → lemma.  
Evaluate exact match rate for Latin vs Unicode source representations.

In [17]:
# ============================================================
# EXPERIMENT 4: LEMMATIZATION (LSTM Seq2Seq)
# ============================================================
exp4_results = {}
MAX_LEN = 50; SEQ_EPOCHS = 50; SEQ_HIDDEN = 256

for lang, df in datasets.items():
    print(f"\n--- Lemmatization: {lang.upper()} ---")
    valid = df[df['lemma'].notna() & (df['lemma'] != '') & (df['lemma'].astype(str) != 'nan')].copy()
    if len(valid) > 20000:
        valid = valid.sample(20000, random_state=SEED)
    print(f"  Subsampled to {len(valid):,}")
    if len(valid) < 100:
        print(f"  Insufficient data ({len(valid)})"); continue
    print(f"  Data: {len(valid):,} entries")
    results = {'lang': lang}

    for repr_name, repr_col in [('latin', 'form_latin'), ('unicode', 'form_unicode')]:
        sources = valid[repr_col].astype(str).tolist()
        targets = valid['lemma'].astype(str).tolist()

        src_chars = set(ch for s in sources for ch in s)
        tgt_chars = set(ch for t in targets for ch in t)
        src2idx = {'<pad>':0, '<sos>':1, '<eos>':2, '<unk>':3}
        for ch in sorted(src_chars): src2idx[ch] = len(src2idx)
        tgt2idx = {'<pad>':0, '<sos>':1, '<eos>':2, '<unk>':3}
        idx2tgt = {0:'<pad>', 1:'<sos>', 2:'<eos>', 3:'<unk>'}
        for ch in sorted(tgt_chars): tgt2idx[ch] = len(tgt2idx); idx2tgt[len(idx2tgt)] = ch

        def enc(text, vocab, ml):
            ids = [vocab.get(ch, vocab['<unk>']) for ch in text[:ml-1]]
            ids.append(vocab['<eos>'])
            return ids + [vocab['<pad>']]*(ml-len(ids))

        X = np.array([enc(s, src2idx, MAX_LEN) for s in sources])
        Y = np.array([enc(t, tgt2idx, MAX_LEN) for t in targets])

        n = len(X); idx = np.random.permutation(n); split = int(0.8*n)
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        X_tr = torch.LongTensor(X[idx[:split]]).to(device)
        Y_tr = torch.LongTensor(Y[idx[:split]]).to(device)
        X_te = torch.LongTensor(X[idx[split:]]).to(device)
        test_tgt = [targets[i] for i in idx[split:]]

        class Seq2Seq(nn.Module):
            def __init__(self):
                super().__init__()
                self.enc_emb = nn.Embedding(len(src2idx), SEQ_HIDDEN)
                self.encoder = nn.LSTM(SEQ_HIDDEN, SEQ_HIDDEN, batch_first=True)
                self.dec_emb = nn.Embedding(len(tgt2idx), SEQ_HIDDEN)
                self.decoder = nn.LSTM(SEQ_HIDDEN, SEQ_HIDDEN, batch_first=True)
                self.fc = nn.Linear(SEQ_HIDDEN, len(tgt2idx))
            def forward(self, src, tgt):
                _, (h, c) = self.encoder(self.enc_emb(src))
                return self.fc(self.decoder(self.dec_emb(tgt), (h, c))[0])
            def predict(self, src, ml=50):
                with torch.no_grad():
                    _, (h, c) = self.encoder(self.enc_emb(src))
                    inp = torch.full((src.size(0),1), 1, dtype=torch.long, device=src.device)
                    outs = []
                    for _ in range(ml):
                        o, (h, c) = self.decoder(self.dec_emb(inp), (h, c))
                        pred = self.fc(o).argmax(dim=-1); outs.append(pred); inp = pred
                    return torch.cat(outs, dim=1)

        model = Seq2Seq().to(device)
        opt = torch.optim.Adam(model.parameters(), lr=0.001)
        crit = nn.CrossEntropyLoss(ignore_index=0)

        model.train()
        bs = 128
        for ep in range(SEQ_EPOCHS):
            perm = torch.randperm(len(X_tr))
            for i in range(0, len(X_tr), bs):
                bi = perm[i:i+bs]; src = X_tr[bi]; tgt = Y_tr[bi]
                tgt_in = torch.cat([torch.full((len(bi),1),1,device=device,dtype=torch.long), tgt[:,:-1]], dim=1)
                loss = crit(model(src, tgt_in).reshape(-1, len(tgt2idx)), tgt.reshape(-1))
                opt.zero_grad(); loss.backward(); opt.step()

        model.eval()
        preds = model.predict(X_te, MAX_LEN).cpu().numpy()
        em = 0
        for i in range(len(preds)):
            chars = []
            for v in preds[i]:
                if v == 2: break
                if v in idx2tgt and v > 3: chars.append(idx2tgt[v])
            if ''.join(chars) == test_tgt[i]: em += 1

        rate = em/len(preds) if preds.size else 0
        results[f'lstm_exact_match_{repr_name}'] = rate
        print(f"  LSTM {repr_name}: {rate:.1%} ({em}/{len(preds)})")

    exp4_results[lang] = results


--- Lemmatization: AKK ---
  Subsampled to 20,000
  Data: 20,000 entries
  LSTM latin: 68.8% (2754/4000)
  LSTM unicode: 64.4% (2577/4000)

--- Lemmatization: SUX ---
  Subsampled to 20,000
  Data: 20,000 entries
  LSTM latin: 81.9% (3277/4000)
  LSTM unicode: 77.5% (3101/4000)

--- Lemmatization: ELX ---
  Subsampled to 1,797
  Data: 1,797 entries
  LSTM latin: 42.5% (153/360)
  LSTM unicode: 25.8% (93/360)


In [23]:
# Train one final model on concatenated features and inspect weights
from scipy.sparse import hstack

for lang, df in datasets.items():
    print(f"\n--- Feature analysis: {lang.upper()} ---")
    counts = df['pos_unified'].value_counts()
    valid = counts[counts >= MIN_CLASS_COUNT].index.tolist()
    task_df = df[df['pos_unified'].isin(valid)].copy()
    if len(task_df) > 50000:
        task_df = task_df.sample(50000, random_state=SEED)

    vec_l = CountVectorizer(analyzer='char', ngram_range=(1,4))
    vec_u = CountVectorizer(analyzer='char', ngram_range=(1,4))
    X_l = vec_l.fit_transform(task_df['form_latin'].astype(str))
    X_u = vec_u.fit_transform(task_df['form_unicode'].astype(str))
    X_concat = hstack([X_l, X_u])
    n_latin_feats = X_l.shape[1]

    clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
    clf.fit(X_concat, task_df['pos_unified'])

    for i, cls in enumerate(clf.classes_):
        weights = clf.coef_[i]
        latin_weight = np.abs(weights[:n_latin_feats]).sum()
        unicode_weight = np.abs(weights[n_latin_feats:]).sum()
        ratio = unicode_weight / (latin_weight + unicode_weight)
        side = "← Unicode" if ratio > 0.55 else ("→ Latin" if ratio < 0.45 else "balanced")
        print(f"  {cls:8s}: Latin={latin_weight:.1f}  Unicode={unicode_weight:.1f}  "
              f"Unicode share={ratio:.1%} {side}")


--- Feature analysis: AKK ---
  ADJ     : Latin=1049.6  Unicode=1043.8  Unicode share=49.9% balanced
  ADP     : Latin=371.2  Unicode=299.0  Unicode share=44.6% → Latin
  ADV     : Latin=461.2  Unicode=443.3  Unicode share=49.0% balanced
  CN      : Latin=354.4  Unicode=173.7  Unicode share=32.9% → Latin
  CONJ    : Latin=93.9  Unicode=49.1  Unicode share=34.3% → Latin
  DET     : Latin=90.4  Unicode=36.6  Unicode share=28.8% → Latin
  DN      : Latin=413.6  Unicode=253.4  Unicode share=38.0% → Latin
  GN      : Latin=530.8  Unicode=358.9  Unicode share=40.3% → Latin
  INTJ    : Latin=231.3  Unicode=172.4  Unicode share=42.7% → Latin
  LN      : Latin=334.1  Unicode=183.4  Unicode share=35.4% → Latin
  MN      : Latin=256.3  Unicode=118.7  Unicode share=31.7% → Latin
  MOD     : Latin=165.2  Unicode=99.7  Unicode share=37.6% → Latin
  NOUN    : Latin=1256.8  Unicode=1279.0  Unicode share=50.4% balanced
  NUM     : Latin=193.5  Unicode=107.9  Unicode share=35.8% → Latin
  ON      : Lat

In [14]:
MIN_CLASS_COUNT = 20
# ============================================================
# ERROR ANALYSIS: Where does each representation fail?
# ============================================================
# Add this cell after the feature weight analysis.
# It trains Latin and Unicode models, collects predictions,
# and categorizes where each wins/fails.
# ============================================================

from scipy.sparse import hstack
from collections import Counter, defaultdict

error_analysis_results = {}

for lang, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  ERROR ANALYSIS — {lang.upper()}")
    print(f"{'='*60}")

    # Use unified POS task
    counts = df['pos_unified'].value_counts()
    valid = counts[counts >= MIN_CLASS_COUNT].index.tolist()
    task_df = df[df['pos_unified'].isin(valid)].copy()
    if len(task_df) > 30000:
        task_df = task_df.sample(30000, random_state=SEED)

    texts_l = task_df['form_latin'].astype(str).values
    texts_u = task_df['form_unicode'].astype(str).values
    labels = task_df['pos_unified'].values

    vec_l = CountVectorizer(analyzer='char', ngram_range=(1, 4))
    vec_u = CountVectorizer(analyzer='char', ngram_range=(1, 4))
    X_l = vec_l.fit_transform(texts_l)
    X_u = vec_u.fit_transform(texts_u)

    # Cross-validated predictions
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    preds_l = np.empty(len(labels), dtype=labels.dtype)
    preds_u = np.empty(len(labels), dtype=labels.dtype)

    for train_idx, test_idx in cv.split(X_l, labels):
        clf_l = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
        clf_u = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
        clf_l.fit(X_l[train_idx], labels[train_idx])
        clf_u.fit(X_u[train_idx], labels[train_idx])
        preds_l[test_idx] = clf_l.predict(X_l[test_idx])
        preds_u[test_idx] = clf_u.predict(X_u[test_idx])

    # Categorize outcomes
    latin_correct = preds_l == labels
    unicode_correct = preds_u == labels

    both_correct = latin_correct & unicode_correct
    both_wrong = ~latin_correct & ~unicode_correct
    latin_only = latin_correct & ~unicode_correct   # Latin right, Unicode wrong
    unicode_only = ~latin_correct & unicode_correct  # Unicode right, Latin wrong

    print(f"\n  Outcome distribution:")
    print(f"    Both correct:       {both_correct.sum():6d} ({both_correct.mean():.1%})")
    print(f"    Both wrong:         {both_wrong.sum():6d} ({both_wrong.mean():.1%})")
    print(f"    Latin only correct: {latin_only.sum():6d} ({latin_only.mean():.1%})")
    print(f"    Unicode only correct:{unicode_only.sum():5d} ({unicode_only.mean():.1%})")

    # ── Analysis 1: By POS category ──
    print(f"\n  Latin vs Unicode advantage by POS:")
    print(f"    {'POS':<10s} {'L-only':>8s} {'U-only':>8s} {'Δ':>8s} {'Winner':>8s}")
    pos_advantage = {}
    for pos in sorted(set(labels)):
        mask = labels == pos
        l_wins = (latin_only & mask).sum()
        u_wins = (unicode_only & mask).sum()
        total = mask.sum()
        delta = l_wins - u_wins
        winner = 'Latin' if delta > 0 else ('Unicode' if delta < 0 else 'tie')
        pos_advantage[pos] = {'latin_wins': l_wins, 'unicode_wins': u_wins,
                              'delta': delta, 'total': total}
        print(f"    {pos:<10s} {l_wins:>8d} {u_wins:>8d} {delta:>+8d} {winner:>8s}")

    # ── Analysis 2: By word length (characters) ──
    print(f"\n  By word length (Latin chars):")
    lengths_l = np.array([len(t) for t in texts_l])
    for lo, hi, label in [(1, 3, '1-3'), (4, 6, '4-6'), (7, 10, '7-10'), (11, 100, '11+')]:
        mask = (lengths_l >= lo) & (lengths_l <= hi)
        if mask.sum() < 10: continue
        l_adv = latin_only[mask].sum()
        u_adv = unicode_only[mask].sum()
        total = mask.sum()
        print(f"    {label:>5s} chars: Latin-only={l_adv:4d}  Unicode-only={u_adv:4d}  "
              f"Δ={l_adv-u_adv:+4d}  (n={total})")

    # ── Analysis 3: By sign count (Unicode signs = spaces + 1) ──
    print(f"\n  By sign count (Unicode):")
    sign_counts = np.array([len(t.split()) for t in texts_u])
    for lo, hi, label in [(1, 1, '1 sign'), (2, 2, '2 signs'), (3, 3, '3 signs'),
                           (4, 5, '4-5'), (6, 100, '6+')]:
        mask = (sign_counts >= lo) & (sign_counts <= hi)
        if mask.sum() < 10: continue
        l_adv = latin_only[mask].sum()
        u_adv = unicode_only[mask].sum()
        total = mask.sum()
        print(f"    {label:>7s}: Latin-only={l_adv:4d}  Unicode-only={u_adv:4d}  "
              f"Δ={l_adv-u_adv:+4d}  (n={total})")

    # ── Analysis 4: Confusion patterns ──
    print(f"\n  Top Latin errors (Latin wrong, Unicode right):")
    uni_wins_mask = unicode_only
    if uni_wins_mask.sum() > 0:
        confusion_pairs = Counter()
        for i in np.where(uni_wins_mask)[0]:
            confusion_pairs[(labels[i], preds_l[i])] += 1
        for (gold, pred), cnt in confusion_pairs.most_common(5):
            print(f"    {gold:>8s} → {pred:<8s} ({cnt}x)")

    print(f"\n  Top Unicode errors (Unicode wrong, Latin right):")
    lat_wins_mask = latin_only
    if lat_wins_mask.sum() > 0:
        confusion_pairs = Counter()
        for i in np.where(lat_wins_mask)[0]:
            confusion_pairs[(labels[i], preds_u[i])] += 1
        for (gold, pred), cnt in confusion_pairs.most_common(5):
            print(f"    {gold:>8s} → {pred:<8s} ({cnt}x)")

    # ── Analysis 5: Concrete examples ──
    print(f"\n  Examples where Unicode wins (Latin wrong):")
    uni_win_idx = np.where(unicode_only)[0]
    if len(uni_win_idx) > 0:
        sample = np.random.choice(uni_win_idx, min(8, len(uni_win_idx)), replace=False)
        for i in sample:
            print(f"    Latin: {texts_l[i]:20s} → pred={preds_l[i]:8s}  "
                  f"Unicode: {texts_u[i]:20s} → pred={preds_u[i]:8s}  "
                  f"gold={labels[i]}")

    print(f"\n  Examples where Latin wins (Unicode wrong):")
    lat_win_idx = np.where(latin_only)[0]
    if len(lat_win_idx) > 0:
        sample = np.random.choice(lat_win_idx, min(8, len(lat_win_idx)), replace=False)
        for i in sample:
            print(f"    Latin: {texts_l[i]:20s} → pred={preds_l[i]:8s}  "
                  f"Unicode: {texts_u[i]:20s} → pred={preds_u[i]:8s}  "
                  f"gold={labels[i]}")

    error_analysis_results[lang] = {
        'both_correct': int(both_correct.sum()),
        'both_wrong': int(both_wrong.sum()),
        'latin_only': int(latin_only.sum()),
        'unicode_only': int(unicode_only.sum()),
        'pos_advantage': pos_advantage,
    }

# ── Summary across languages ──
print(f"\n{'='*60}")
print(f"  ERROR ANALYSIS SUMMARY")
print(f"{'='*60}")
print(f"\n  {'Lang':<6s} {'Both OK':>10s} {'Both Err':>10s} {'Latin Only':>12s} {'Uni Only':>12s} {'Complementary':>14s}")
for lang in ['akk', 'sux', 'elx']:
    if lang not in error_analysis_results: continue
    r = error_analysis_results[lang]
    total = r['both_correct'] + r['both_wrong'] + r['latin_only'] + r['unicode_only']
    complementary = r['latin_only'] + r['unicode_only']
    print(f"  {lang.upper():<6s} {r['both_correct']:>10d} {r['both_wrong']:>10d} "
          f"{r['latin_only']:>12d} {r['unicode_only']:>12d} {complementary:>14d} ({complementary/total:.1%})")


  ERROR ANALYSIS — AKK

  Outcome distribution:
    Both correct:        18935 (63.1%)
    Both wrong:           5542 (18.5%)
    Latin only correct:   3710 (12.4%)
    Unicode only correct: 1813 (6.0%)

  Latin vs Unicode advantage by POS:
    POS          L-only   U-only        Δ   Winner
    ADJ             156       51     +105    Latin
    ADP             290       11     +279    Latin
    ADV              35        8      +27    Latin
    CN               24        4      +20    Latin
    CONJ             43        0      +43    Latin
    DET               1        3       -2  Unicode
    DN              157        9     +148    Latin
    GN               47        9      +38    Latin
    INTJ             12        2      +10    Latin
    LN                5        3       +2    Latin
    MN               11        2       +9    Latin
    MOD              27        1      +26    Latin
    NOUN           2015     1378     +637    Latin
    NUM              94       11      +83   

## Experiment 5: LLM Evaluation

Prepare test sets and prompts for Claude + GPT-4o (zero-shot and 5-shot).  
Run API calls separately with your keys.

In [18]:
# ============================================================
# EXPERIMENT 5: LLM EVALUATION — DATA PREP
# ============================================================
exp5_data = {}

for lang, df in datasets.items():
    print(f"\n--- LLM Eval Prep: {lang.upper()} ---")
    pos_col = 'pos_unified'
    pos_counts = df[pos_col].value_counts()
    valid_pos = pos_counts[pos_counts >= 20].index.tolist()

    # Few-shot examples (5 per class)
    few_shot = []
    for pos in valid_pos:
        subset = df[df[pos_col] == pos]
        sample = subset.sample(min(5, len(subset)), random_state=SEED)
        for _, row in sample.iterrows():
            few_shot.append({
                'form_latin': str(row['form_latin']),
                'form_unicode': str(row.get('form_unicode', '')),
                'pos': pos, 'lemma': str(row.get('lemma', '')),
            })

    # Test set (200 items, held out)
    test_pool = df[df[pos_col].isin(valid_pos)]
    test_df = test_pool.sample(min(200, len(test_pool)), random_state=SEED+1)
    test_items = [{
        'form_latin': str(row['form_latin']),
        'form_unicode': str(row.get('form_unicode', '')),
        'gold_pos': row[pos_col],
        'gold_lemma': str(row.get('lemma', '')),
    } for _, row in test_df.iterrows()]

    lang_name = {'akk': 'Akkadian', 'sux': 'Sumerian', 'elx': 'Elamite'}.get(lang, lang)
    prompt = f"""You are an expert in ancient {lang_name} cuneiform.
Given the following word in {{representation}}, classify its part of speech.
Valid POS tags: {valid_pos}

{{few_shot_section}}

Word: {{word}}
POS:"""

    exp5_data[lang] = {
        'few_shot': few_shot, 'test_items': test_items,
        'prompt': prompt, 'valid_pos': valid_pos,
    }
    print(f"  {len(few_shot)} few-shot examples, {len(test_items)} test items")
    print(f"  Valid POS: {valid_pos}")


--- LLM Eval Prep: AKK ---
  120 few-shot examples, 200 test items
  Valid POS: ['NOUN', 'VERB', 'ADP', 'NUM', 'PN', 'X', 'ADJ', 'DET', 'DN', 'CONJ', 'CN', 'MOD', 'REL', 'GN', 'INTJ', 'ADV', 'MN', 'RN', 'LN', 'TN', 'SBJN', 'ON', 'WN', 'QN']

--- LLM Eval Prep: SUX ---
  70 few-shot examples, 200 test items
  Valid POS: ['NOUN', 'VERB', 'DN', 'NUM', 'SN', 'INTJ', 'X', 'RN', 'ADJ', 'QP', 'TN', 'GN', 'PN', 'WN']

--- LLM Eval Prep: ELX ---
  35 few-shot examples, 200 test items
  Valid POS: ['PN', 'NOUN', 'VERB', 'GN', 'OTHER', 'ADJ', 'DN']


In [20]:
# ============================================================
# EXPERIMENT 5: RUN LLM API CALLS (uncomment & add keys)
# ============================================================

# import anthropic, openai

# def run_llm_pos(client, model_name, word, representation, prompt_template,
#                 few_shot_examples=None, repr_key='form_latin'):
#     few_shot_section = ""
#     if few_shot_examples:
#         lines = [f"Word: {ex[repr_key]} → POS: {ex['pos']}" for ex in few_shot_examples]
#         few_shot_section = "Examples:\n" + "\n".join(lines)

#     prompt = prompt_template.replace("{representation}", representation)
#     prompt = prompt.replace("{few_shot_section}", few_shot_section)
#     prompt = prompt.replace("{word}", word)

#     # Call Claude or GPT-4o...
#     pass

# # Example loop:
# for lang, data in exp5_data.items():
#     for item in data['test_items']:
#         for shot in ['zero', 'five']:
#             for repr_key in ['form_latin', 'form_unicode']:
#                 few_shot = data['few_shot'] if shot == 'five' else None
#                 pred = run_llm_pos(client, model, item[repr_key],
#                                   repr_key.split('_')[1], data['prompt'], few_shot, repr_key)
#print("LLM evaluation data prepared. Uncomment API code above to run.")

## Experiment 6: Gated Dual Encoder

In [9]:
# ============================================================
# GATED DUAL-ENCODER: Learns per-input representation weighting
# ============================================================
# Adds methodological contribution: a simple model that learns
# WHEN to trust Latin vs Unicode for each input.
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
import numpy as np

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

MIN_CLASS_COUNT = 20


class GatedDualEncoder(nn.Module):
    """
    Dual-encoder with learned gating between Latin and Unicode representations.

    Each input is encoded by two separate character-level BiLSTMs.
    A learned gate determines per-input how much to trust each representation:

        h_final = g * h_latin + (1 - g) * h_unicode
        g = sigmoid(W_gate · [h_latin; h_unicode] + b_gate)

    The gate value g is interpretable: g → 1 means Latin-dominant,
    g → 0 means Unicode-dominant.
    """
    def __init__(self, vocab_size_l, vocab_size_u, embed_dim, hidden_dim, n_classes, dropout=0.3):
        super().__init__()
        # Latin encoder
        self.embed_l = nn.Embedding(vocab_size_l, embed_dim, padding_idx=0)
        self.lstm_l = nn.LSTM(embed_dim, hidden_dim, num_layers=2,
                              bidirectional=True, batch_first=True, dropout=dropout)

        # Unicode encoder
        self.embed_u = nn.Embedding(vocab_size_u, embed_dim, padding_idx=0)
        self.lstm_u = nn.LSTM(embed_dim, hidden_dim, num_layers=2,
                              bidirectional=True, batch_first=True, dropout=dropout)

        # Gate: takes concatenated hidden states, outputs scalar per sample
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

        # Classifier
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, n_classes)

    def encode(self, x, embed, lstm):
        emb = self.dropout(embed(x))
        _, (h, _) = lstm(emb)
        return torch.cat([h[-2], h[-1]], dim=1)  # concat forward/backward

    def forward(self, x_latin, x_unicode):
        h_l = self.encode(x_latin, self.embed_l, self.lstm_l)
        h_u = self.encode(x_unicode, self.embed_u, self.lstm_u)

        # Compute gate
        g = self.gate(torch.cat([h_l, h_u], dim=1))  # (batch, 1)

        # Gated combination
        h_combined = g * h_l + (1 - g) * h_u

        return self.fc(self.dropout(h_combined)), g


def build_char_vocab(texts):
    char2idx = {'<pad>': 0, '<unk>': 1}
    for t in texts:
        for ch in str(t):
            if ch not in char2idx:
                char2idx[ch] = len(char2idx)
    return char2idx


def encode_texts(texts, char2idx, max_len):
    X = np.zeros((len(texts), max_len), dtype=np.int64)
    for i, t in enumerate(texts):
        for j, ch in enumerate(str(t)[:max_len]):
            X[i, j] = char2idx.get(ch, 1)
    return X


# ============================================================
# RUN GATED MODEL ACROSS ALL LANGUAGES AND TASKS
# ============================================================

EMBED_DIM = 64
HIDDEN_DIM = 64
DROPOUT = 0.3
LR = 0.001
BATCH_SIZE = 64
N_EPOCHS = 25
MAX_LEN = 80

gated_results = {}

for lang, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  GATED DUAL-ENCODER — {lang.upper()}")
    print(f"{'='*60}")

    for pos_col, task_name in [('pos_unified', 'unified_pos'),
                                ('pos_grammatical', 'gram_pos')]:
        counts = df[pos_col].value_counts()
        valid = counts[counts >= MIN_CLASS_COUNT].index.tolist()
        task_df = df[df[pos_col].isin(valid)].copy()

        if len(task_df) > 10000:
            task_df = task_df.sample(10000, random_state=SEED)

        if len(valid) < 2:
            continue

        texts_l = task_df['form_latin'].astype(str).tolist()
        texts_u = task_df['form_unicode'].astype(str).tolist()

        le = LabelEncoder()
        labels = le.fit_transform(task_df[pos_col])
        n_classes = len(le.classes_)

        # Build vocabs
        vocab_l = build_char_vocab(texts_l)
        vocab_u = build_char_vocab(texts_u)

        X_l = encode_texts(texts_l, vocab_l, MAX_LEN)
        X_u = encode_texts(texts_u, vocab_u, MAX_LEN)

        # Cross-validation
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        fold_scores = []
        all_gates = []
        all_labels_for_gates = []
        all_sign_counts = []

        for fold, (train_idx, val_idx) in enumerate(cv.split(X_l, labels)):
            Xl_tr = torch.LongTensor(X_l[train_idx]).to(device)
            Xu_tr = torch.LongTensor(X_u[train_idx]).to(device)
            y_tr = torch.LongTensor(labels[train_idx]).to(device)

            Xl_va = torch.LongTensor(X_l[val_idx]).to(device)
            Xu_va = torch.LongTensor(X_u[val_idx]).to(device)
            y_va = labels[val_idx]

            model = GatedDualEncoder(
                len(vocab_l), len(vocab_u), EMBED_DIM, HIDDEN_DIM, n_classes, DROPOUT
            ).to(device)

            optimizer = torch.optim.Adam(model.parameters(), lr=LR)
            # Class weights
            wts = torch.FloatTensor(
                [1.0 / max((labels[train_idx] == c).sum(), 1) for c in range(n_classes)]
            ).to(device)
            criterion = nn.CrossEntropyLoss(weight=wts)

            # Train
            train_ds = TensorDataset(Xl_tr, Xu_tr, y_tr)
            loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

            model.train()
            for epoch in range(N_EPOCHS):
                for xl_b, xu_b, yb in loader:
                    optimizer.zero_grad()
                    logits, g = model(xl_b, xu_b)
                    loss = criterion(logits, yb)
                    loss.backward()
                    optimizer.step()

            # Evaluate
            model.eval()
            with torch.no_grad():
                all_preds, all_g = [], []
                for k in range(0, len(Xl_va), 256):
                    logits, g = model(Xl_va[k:k+256], Xu_va[k:k+256])
                    all_preds.append(logits.argmax(dim=1).cpu().numpy())
                    all_g.append(g.cpu().numpy())

                preds = np.concatenate(all_preds)
                gates = np.concatenate(all_g).flatten()

            f1 = f1_score(y_va, preds, average='macro')
            fold_scores.append(f1)

            # Collect gate values for analysis
            all_gates.extend(gates)
            all_labels_for_gates.extend(le.inverse_transform(y_va))
            all_sign_counts.extend([len(texts_u[i].split()) for i in val_idx])

            del model, Xl_tr, Xu_tr, Xl_va, Xu_va
            torch.cuda.empty_cache()

        mean_f1 = np.mean(fold_scores)
        print(f"\n  {task_name}: F1 = {mean_f1:.4f} (±{np.std(fold_scores):.4f})")

        gated_results.setdefault(lang, {})[task_name] = mean_f1

        # ── Gate Analysis: by POS category ──
        print(f"\n  Gate values by POS (g→1 = Latin, g→0 = Unicode):")
        gates_arr = np.array(all_gates)
        labels_arr = np.array(all_labels_for_gates)
        signs_arr = np.array(all_sign_counts)

        for pos in sorted(set(labels_arr)):
            mask = labels_arr == pos
            if mask.sum() >= 10:
                mean_g = gates_arr[mask].mean()
                direction = "← Latin" if mean_g > 0.55 else ("→ Unicode" if mean_g < 0.45 else "balanced")
                print(f"    {pos:<10s}: g={mean_g:.3f} {direction}  (n={mask.sum()})")

        # ── Gate Analysis: by sign count ──
        print(f"\n  Gate values by sign count:")
        for lo, hi, label in [(1, 1, '1 sign'), (2, 2, '2 signs'), (3, 3, '3 signs'),
                               (4, 5, '4-5'), (6, 100, '6+')]:
            mask = (signs_arr >= lo) & (signs_arr <= hi)
            if mask.sum() >= 10:
                mean_g = gates_arr[mask].mean()
                print(f"    {label:>7s}: g={mean_g:.3f}  (n={mask.sum()})")


# ── Comparison table ──
print(f"\n{'='*60}")
print(f"  MODEL COMPARISON")
print(f"{'='*60}")
print(f"\n  {'Task':<15s} {'Lang':<6s} {'Latin':>8s} {'Unicode':>8s} {'Concat':>8s} {'Gated':>8s}")
print(f"  {'-'*15} {'-'*6} {'-'*8} {'-'*8} {'-'*8} {'-'*8}")

# You'll need to have exp3c_results available, or manually enter the values
for lang in ['akk', 'sux', 'elx']:
    if lang not in gated_results:
        continue
    for task in ['unified_pos', 'gram_pos']:
        gated_f1 = gated_results[lang].get(task, 0)
        print(f"  {task:<15s} {lang.upper():<6s} {'---':>8s} {'---':>8s} {'---':>8s} {gated_f1:>8.4f}")

Device: cuda

  GATED DUAL-ENCODER — AKK

  unified_pos: F1 = 0.5090 (±0.0336)

  Gate values by POS (g→1 = Latin, g→0 = Unicode):
    ADJ       : g=1.000 ← Latin  (n=464)
    ADP       : g=1.000 ← Latin  (n=1001)
    ADV       : g=1.000 ← Latin  (n=85)
    CN        : g=1.000 ← Latin  (n=145)
    CONJ      : g=1.000 ← Latin  (n=197)
    DET       : g=0.804 ← Latin  (n=311)
    DN        : g=1.000 ← Latin  (n=215)
    GN        : g=1.000 ← Latin  (n=110)
    INTJ      : g=1.000 ← Latin  (n=78)
    LN        : g=1.000 ← Latin  (n=49)
    MN        : g=1.000 ← Latin  (n=88)
    MOD       : g=1.000 ← Latin  (n=153)
    NOUN      : g=1.000 ← Latin  (n=3944)
    NUM       : g=1.000 ← Latin  (n=819)
    PN        : g=1.000 ← Latin  (n=537)
    REL       : g=0.820 ← Latin  (n=164)
    RN        : g=1.000 ← Latin  (n=59)
    TN        : g=1.000 ← Latin  (n=36)
    VERB      : g=1.000 ← Latin  (n=1049)
    X         : g=0.998 ← Latin  (n=472)

  Gate values by sign count:
     1 sign: g=0.979  

## Experiment 6: Baseline Transformer

In [10]:
# ============================================================
# CHARACTER TRANSFORMER BASELINE
# ============================================================
# Tests whether the complementarity effect persists under
# a modern neural architecture (not just linear LR).
#
# Simple 2-layer character Transformer encoder trained from
# scratch. Runs Latin vs Unicode vs Concat on unified POS
# and grammatical POS for all three languages.
#
# Prerequisites: cells 1-13 (data loading), MIN_CLASS_COUNT=20
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
import numpy as np
import math

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

MIN_CLASS_COUNT = 20


class CharTransformerClassifier(nn.Module):
    """
    Simple character-level Transformer encoder for text classification.
    2-layer, 4-head, trained from scratch.
    """
    def __init__(self, vocab_size, n_classes, embed_dim=128, n_heads=4,
                 n_layers=2, max_len=100, dropout=0.3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embed = nn.Embedding(max_len, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads, dim_feedforward=256,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(embed_dim, n_classes)
        self.embed_dim = embed_dim

    def forward(self, x):
        # x: (batch, seq_len)
        mask = (x == 0)  # padding mask

        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        emb = self.embed(x) * math.sqrt(self.embed_dim) + self.pos_embed(positions)
        emb = self.dropout(emb)

        out = self.transformer(emb, src_key_padding_mask=mask)

        # Mean pooling over non-padding tokens
        mask_expanded = (~mask).unsqueeze(-1).float()
        pooled = (out * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)

        return self.fc(self.dropout(pooled))


def build_vocab(texts):
    char2idx = {'<pad>': 0, '<unk>': 1, '<sep>': 2}
    for t in texts:
        for ch in str(t):
            if ch not in char2idx:
                char2idx[ch] = len(char2idx)
    return char2idx


def encode(texts, vocab, max_len):
    X = np.zeros((len(texts), max_len), dtype=np.int64)
    for i, t in enumerate(texts):
        for j, ch in enumerate(str(t)[:max_len]):
            X[i, j] = vocab.get(ch, 1)
    return X


def run_transformer_cv(X, labels_enc, n_classes, vocab_size, lang, repr_name,
                        max_len=100, n_splits=5, n_epochs=20, batch_size=64, lr=0.001):
    """Train and evaluate character Transformer with cross-validation."""
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X, labels_enc)):
        X_tr = torch.LongTensor(X[train_idx]).to(device)
        y_tr = torch.LongTensor(labels_enc[train_idx]).to(device)
        X_va = torch.LongTensor(X[val_idx]).to(device)
        y_va = labels_enc[val_idx]

        model = CharTransformerClassifier(
            vocab_size=vocab_size, n_classes=n_classes,
            embed_dim=128, n_heads=4, n_layers=2, max_len=max_len, dropout=0.3
        ).to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

        wts = torch.FloatTensor(
            [1.0 / max((labels_enc[train_idx] == c).sum(), 1) for c in range(n_classes)]
        ).to(device)
        criterion = nn.CrossEntropyLoss(weight=wts)

        loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)

        model.train()
        for epoch in range(n_epochs):
            for xb, yb in loader:
                optimizer.zero_grad()
                loss = criterion(model(xb), yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()

        model.eval()
        with torch.no_grad():
            all_preds = []
            for k in range(0, len(X_va), 256):
                preds = model(X_va[k:k+256]).argmax(dim=1).cpu().numpy()
                all_preds.append(preds)
            preds = np.concatenate(all_preds)

        f1 = f1_score(y_va, preds, average='macro')
        fold_scores.append(f1)

        del model, X_tr, y_tr, X_va
        torch.cuda.empty_cache()

    mean_f1 = np.mean(fold_scores)
    std_f1 = np.std(fold_scores)
    print(f"    {repr_name}: {mean_f1:.4f} (±{std_f1:.4f})")
    return mean_f1


# ============================================================
# RUN TRANSFORMER ON ALL LANGUAGES
# ============================================================

MAX_LEN = 80
N_EPOCHS = 20
transformer_results = {}

for lang, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  CHAR TRANSFORMER — {lang.upper()}")
    print(f"{'='*60}")

    for pos_col, task_name in [('pos_unified', 'unified_pos'),
                                ('pos_grammatical', 'gram_pos')]:
        counts = df[pos_col].value_counts()
        valid = counts[counts >= MIN_CLASS_COUNT].index.tolist()
        task_df = df[df[pos_col].isin(valid)].copy()

        if len(task_df) > 10000:
            task_df = task_df.sample(10000, random_state=SEED)

        if len(valid) < 2:
            continue

        print(f"\n  {task_name} ({len(task_df):,} tokens, {len(valid)} classes)")

        le = LabelEncoder()
        labels_enc = le.fit_transform(task_df[pos_col])
        n_classes = len(le.classes_)

        texts_l = task_df['form_latin'].astype(str).tolist()
        texts_u = task_df['form_unicode'].astype(str).tolist()
        # Concat: "latin_text [SEP] unicode_text"
        texts_c = [f"{l} \x00 {u}" for l, u in zip(texts_l, texts_u)]

        # Build combined vocab (covers all three conditions)
        all_texts = texts_l + texts_u + texts_c
        vocab = build_vocab(all_texts)

        X_l = encode(texts_l, vocab, MAX_LEN)
        X_u = encode(texts_u, vocab, MAX_LEN)
        X_c = encode(texts_c, vocab, min(MAX_LEN * 2, 160))

        # Latin
        f1_l = run_transformer_cv(X_l, labels_enc, n_classes, len(vocab),
                                   lang, 'Latin', MAX_LEN, n_epochs=N_EPOCHS)

        # Unicode
        f1_u = run_transformer_cv(X_u, labels_enc, n_classes, len(vocab),
                                   lang, 'Unicode', MAX_LEN, n_epochs=N_EPOCHS)

        # Concat
        f1_c = run_transformer_cv(X_c, labels_enc, n_classes, len(vocab),
                                   lang, 'Concat', min(MAX_LEN * 2, 160), n_epochs=N_EPOCHS)

        best_single = max(f1_l, f1_u)
        gain = f1_c - best_single
        sig = '***' if gain > 0.01 else ('+' if gain > 0 else '-')

        transformer_results.setdefault(lang, {})[task_name] = {
            'latin': f1_l, 'unicode': f1_u, 'concat': f1_c,
            'gain': gain, 'sig': sig
        }

        print(f"    Concat gain: {gain:+.4f} {sig}")

# ============================================================
# SUMMARY COMPARISON: LR vs Transformer
# ============================================================
print(f"\n{'='*60}")
print(f"  TRANSFORMER vs LR COMPARISON")
print(f"{'='*60}")
print(f"\n  Does concatenation complementarity persist under a Transformer?")
print(f"\n  {'Task':<15s} {'Lang':<5s} {'LR-L':>7s} {'LR-U':>7s} {'LR-C':>7s} {'TF-L':>7s} {'TF-U':>7s} {'TF-C':>7s} {'LR Δ':>7s} {'TF Δ':>7s}")
print(f"  {'-'*15} {'-'*5} {'-'*7} {'-'*7} {'-'*7} {'-'*7} {'-'*7} {'-'*7} {'-'*7} {'-'*7}")

# LR results (enter your actual values here or pull from exp3c_results)
lr_results = {
    'akk': {
        'unified_pos': {'latin': 0.693, 'unicode': 0.621, 'concat': 0.712},
        'gram_pos': {'latin': 0.669, 'unicode': 0.657, 'concat': 0.682},
    },
    'sux': {
        'unified_pos': {'latin': 0.847, 'unicode': 0.750, 'concat': 0.870},
        'gram_pos': {'latin': 0.872, 'unicode': 0.832, 'concat': 0.888},
    },
    'elx': {
        'unified_pos': {'latin': 0.636, 'unicode': 0.659, 'concat': 0.666},
        'gram_pos': {'latin': 0.615, 'unicode': 0.634, 'concat': 0.642},
    },
}

concat_wins = 0
total_comparisons = 0

for lang in ['akk', 'sux', 'elx']:
    if lang not in transformer_results:
        continue
    for task in ['unified_pos', 'gram_pos']:
        if task not in transformer_results[lang]:
            continue

        tf = transformer_results[lang][task]
        lr = lr_results.get(lang, {}).get(task, {})

        if not lr:
            continue

        lr_gain = lr['concat'] - max(lr['latin'], lr['unicode'])
        tf_gain = tf['concat'] - max(tf['latin'], tf['unicode'])

        total_comparisons += 1
        if tf_gain > 0:
            concat_wins += 1

        print(f"  {task:<15s} {lang.upper():<5s} "
              f"{lr['latin']:>7.3f} {lr['unicode']:>7.3f} {lr['concat']:>7.3f} "
              f"{tf['latin']:>7.3f} {tf['unicode']:>7.3f} {tf['concat']:>7.3f} "
              f"{lr_gain:>+7.3f} {tf_gain:>+7.3f}")

print(f"\n  Concat wins under Transformer: {concat_wins}/{total_comparisons}")
if concat_wins == total_comparisons:
    print(f"  → Complementarity effect persists across architectures!")
elif concat_wins > total_comparisons // 2:
    print(f"  → Complementarity effect mostly persists.")
else:
    print(f"  → Complementarity effect may be architecture-dependent.")

Device: cuda

  CHAR TRANSFORMER — AKK

  unified_pos (10,000 tokens, 24 classes)
    Latin: 0.3605 (±0.0193)
    Unicode: 0.3730 (±0.0218)
    Concat: 0.4122 (±0.0247)
    Concat gain: +0.0392 ***

  gram_pos (10,000 tokens, 14 classes)
    Latin: 0.5162 (±0.0138)
    Unicode: 0.4700 (±0.0188)
    Concat: 0.5129 (±0.0138)
    Concat gain: -0.0033 -

  CHAR TRANSFORMER — SUX

  unified_pos (10,000 tokens, 14 classes)
    Latin: 0.3799 (±0.0192)
    Unicode: 0.4281 (±0.0137)
    Concat: 0.4507 (±0.0181)
    Concat gain: +0.0226 ***

  gram_pos (10,000 tokens, 7 classes)
    Latin: 0.5612 (±0.0122)
    Unicode: 0.6301 (±0.0189)
    Concat: 0.6538 (±0.0206)
    Concat gain: +0.0238 ***

  CHAR TRANSFORMER — ELX

  unified_pos (10,000 tokens, 7 classes)
    Latin: 0.4462 (±0.0082)
    Unicode: 0.4713 (±0.0178)
    Concat: 0.4891 (±0.0103)
    Concat gain: +0.0177 ***

  gram_pos (10,000 tokens, 5 classes)
    Latin: 0.4302 (±0.0062)
    Unicode: 0.4649 (±0.0120)
    Concat: 0.4687 (±0.0101

## Results Summary & LaTeX Tables

In [21]:
# ============================================================
# COMPILE ALL RESULTS
# ============================================================
all_results = {
    'exp1': exp1_results,
    'exp1_graph': exp1_graph_results,
    'exp2': exp2_results,
    'exp3': exp3_results,
    'exp3_bilstm': exp3_bilstm_results,
    'exp4': exp4_results,
    'exp5_data': {k: {'n_test': len(v['test_items']), 'valid_pos': v['valid_pos']}
                  for k, v in exp5_data.items()},
}

# Save
with open(BASE_PATH + 'three_lang_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
print("Results saved to three_lang_results.json")

# ── Print summary table ──
print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)

langs = [l for l in ['elx', 'akk', 'sux'] if l in datasets]
header = f"{'Metric':<40s} " + " ".join(f"{l.upper():>10s}" for l in langs)
print(header)
print("-"*len(header))

# Exp 1
for metric in ['silhouette_latin', 'silhouette_unicode', 'morpheme_coherence_latin', 'morpheme_coherence_unicode']:
    vals = []
    for l in langs:
        v = exp1_results.get(l, {}).get(metric, None)
        vals.append(f"{v:.4f}" if v is not None else "-")
    print(f"Exp1 {metric:<34s} " + " ".join(f"{v:>10s}" for v in vals))

# Exp 2
for metric in ['tp_f1', 'tp_precision', 'tp_recall']:
    vals = []
    for l in langs:
        v = exp2_results.get(l, {}).get(metric, None)
        vals.append(f"{v:.4f}" if v is not None else "-")
    print(f"Exp2 {metric:<34s} " + " ".join(f"{v:>10s}" for v in vals))

# Exp 3
for key_pattern in ['ngram_lr_unified_latin', 'ngram_lr_unified_unicode',
                     'ngram_lr_grammatical_latin', 'ngram_lr_grammatical_unicode',
                     'knn_unified_latin', 'knn_unified_unicode']:
    vals = []
    for l in langs:
        v = exp3_results.get(l, {}).get(key_pattern, None)
        vals.append(f"{v:.4f}" if v is not None else "-")
    print(f"Exp3 {key_pattern:<34s} " + " ".join(f"{v:>10s}" for v in vals))

# Exp 4
for metric in ['lstm_exact_match_latin', 'lstm_exact_match_unicode']:
    vals = []
    for l in langs:
        v = exp4_results.get(l, {}).get(metric, None)
        vals.append(f"{v:.4f}" if v is not None else "-")
    print(f"Exp4 {metric:<34s} " + " ".join(f"{v:>10s}" for v in vals))

TypeError: keys must be str, int, float, bool or None, not tuple

In [22]:
# ============================================================
# LATEX TABLES
# ============================================================
langs = [l for l in ['elx', 'akk', 'sux'] if l in datasets]
lang_names = {'elx': 'Elamite', 'akk': 'Akkadian', 'sux': 'Sumerian'}

print("% Table: Embedding Comparison")
print(r"\begin{table}[t]")
print(r"\centering")
print(r"\small")
print(r"\begin{tabular}{ll" + "c"*len(langs) + "}")
print(r"\toprule")
print("Metric & Repr. & " + " & ".join(lang_names.get(l,l) for l in langs) + r" \\")
print(r"\midrule")
for metric in ['silhouette', 'morpheme_coherence']:
    for rn in ['latin', 'unicode']:
        vals = []
        for l in langs:
            v = exp1_results.get(l, {}).get(f'{metric}_{rn}', None)
            vals.append(f"{v:.3f}" if isinstance(v, float) else "--")
        label = metric.replace('_',' ').title()
        print(f"{label} & {rn.title()} & " + " & ".join(vals) + r" \\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\caption{Embedding comparison across three cuneiform languages.}")
print(r"\label{tab:embeddings}")
print(r"\end{table}")

print("\n% Table: POS Classification")
print(r"\begin{table}[t]")
print(r"\centering")
print(r"\small")
print(r"\begin{tabular}{lll" + "c"*len(langs) + "}")
print(r"\toprule")
print("Model & Tagset & Repr. & " + " & ".join(lang_names.get(l,l) for l in langs) + r" \\")
print(r"\midrule")
for model_name, prefix in [('N-gram LR', 'ngram_lr'), ('k-NN', 'knn')]:
    for ts in ['unified', 'grammatical']:
        for rn in ['latin', 'unicode']:
            vals = []
            for l in langs:
                v = exp3_results.get(l, {}).get(f'{prefix}_{ts}_{rn}', None)
                vals.append(f"{v:.3f}" if isinstance(v, float) else "--")
            print(f"{model_name} & {ts} & {rn} & " + " & ".join(vals) + r" \\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\caption{POS classification F1-macro.}")
print(r"\label{tab:pos}")
print(r"\end{table}")

% Table: Embedding Comparison
\begin{table}[t]
\centering
\small
\begin{tabular}{llccc}
\toprule
Metric & Repr. & Elamite & Akkadian & Sumerian \\
\midrule
Silhouette & Latin & -- & -- & -- \\
Silhouette & Unicode & -- & -- & -- \\
Morpheme Coherence & Latin & -- & -- & -- \\
Morpheme Coherence & Unicode & -- & -- & -- \\
\bottomrule
\end{tabular}
\caption{Embedding comparison across three cuneiform languages.}
\label{tab:embeddings}
\end{table}

% Table: POS Classification
\begin{table}[t]
\centering
\small
\begin{tabular}{lllccc}
\toprule
Model & Tagset & Repr. & Elamite & Akkadian & Sumerian \\
\midrule
N-gram LR & unified & latin & 0.636 & 0.693 & 0.847 \\
N-gram LR & unified & unicode & 0.659 & 0.621 & 0.750 \\
N-gram LR & grammatical & latin & 0.605 & 0.665 & 0.881 \\
N-gram LR & grammatical & unicode & 0.632 & 0.643 & 0.822 \\
k-NN & unified & latin & 0.530 & 0.707 & 0.837 \\
k-NN & unified & unicode & 0.526 & 0.584 & 0.760 \\
k-NN & grammatical & latin & 0.521 & 0.716 & 0.856 \

## Paper Outline

**Title:** Does Script Representation Matter? Evidence from Three Cuneiform Languages

1. **Introduction** (1p) — First controlled comparison of Latin vs Unicode across 3 languages, 5 tasks
2. **Background** (1p) — Cuneiform writing system, representation choices, related work
3. **Data & Preprocessing** (1p) — Datasets, Unicode conversion, POS harmonization
4. **Experiments** (3p) — Embeddings, word boundaries, POS, lemmatization, LLM eval
5. **Analysis & Discussion** (1p) — Complementarity explanation, scale effects, recommendations
6. **Conclusion** (0.5p) — Neither dominates; word boundaries from raw Unicode; benchmark release

**Figures:** t-SNE plots, per-class silhouette bars, TP threshold curves, radar chart  
**Tables:** Dataset stats, embedding metrics, word boundaries, POS F1, lemmatization, LLM comparison